# Superwindow plasticity — FINAL FIGURE

Locked analysis notebook for the paper figure.

**Default (set1):** multi-length amalgamation  
`mean_per_superwindow` + `equal_length_mean` + all lengths.

**Optional:** original fixed-length path from `AAA_SUPERWINDOW_PLASTICITY_NO_DTW_2026.ipynb`  
(`SUPERWINDOW_MODE = "fixed_length"`, `FIXED_SUPERWINDOW_LENGTH = 15`).

Pipeline:

1. Load windowed + instant predictions  
2. Filename exclusions  
3. Build superwindows (multi-length **or** fixed L=15)  
4. FILTERS → analysis subset  
5. Estimate switches under locked settings  
6. Final figures: fans, heatmaps, direct/indirect bar plots  

Toggle everything important in **cell 0 / cell 3 / cell 5**.


**Data set** (`DATASET` in the filters cell):

- `"HTT_neurons"` (default, the paper figure): HTT, ventral neurons, growth cone, the 7 HTT genotypes, the 4 HTT
  groups, no nocodazole. Control 20H20S.
- `"ES_cells"`: ES cells, genotypes RUES2 / HTT72CAG / HTTKO. Control RUES2. The dose-series panels
  (direct vs indirect, 56/77/150) only make sense for the neurons.

Bad movies are excluded by filename in "2. Manual filename exclusions".

## Locked defaults

| Knob | Set1 (default) | Fixed-length option |
|------|----------------|---------------------|
| `SUPERWINDOW_MODE` | `"multi_length"` | `"fixed_length"` |
| Lengths | `list(range(4, 31, 2))` (or denser) | `[15]` |
| `SW_UNIT_POLICY` | `mean_per_superwindow` | same (matches NO_DTW) |
| `LENGTH_AGG_POLICY` | `equal_length_mean` | ignored (single L) |
| `MAX_SW_PER_TRACK` | `5` | `5` (no-op unless `cap_k...`) |
| Quantity | `conditional` | same |
| Display | `delta_pct_vs_control_stabilized`, eps=`0.1` | same |
| Control | `20H20S` | same |

Filters match the HTT GC ventral analysis used throughout.

In [ ]:
# =============================================================================
# 0. Imports, paths, run switches
# =============================================================================
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from matplotlib.colors import to_hex

sys.path.insert(0, "..")
from sptnano import config, movies, plasticity, raw_images, superwindows


pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(50)
pl.Config.set_tbl_width_chars(180)
plt.style.use("seaborn-v0_8-whitegrid")

paths = config.load_paths()
TRANSFORMER_BEHAVIOR_DIR = paths["evaluation_dir"]
TRANSFORMER_WINDOWED_PARQUET = TRANSFORMER_BEHAVIOR_DIR / "heldout_test_windows_with_transformer_behavior_state.parquet"
TRANSFORMER_INSTANT_PARQUET = TRANSFORMER_BEHAVIOR_DIR / "heldout_test_instant_with_transformer_behavior_state.parquet"

PLOTS_DIR = paths["plots_dir"]
RESULTS_DIR = PLOTS_DIR / "superwindow_plasticity_final_fig_2026"
SUPERWINDOW_SAVE_DIR = RESULTS_DIR / "superwindow_dfs"
PLASTICITY_PLOTS_DIR = RESULTS_DIR / "plots_plasticity"
for path in [RESULTS_DIR, SUPERWINDOW_SAVE_DIR, PLASTICITY_PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FRAME_INTERVAL = float(config.TIME_BETWEEN_FRAMES)
TIME_WINDOW_FRAMES = int(config.TIME_WINDOW)
WINDOW_OVERLAP_FRAMES = int(config.OVERLAP)
WINDOW_STEP_FRAMES = TIME_WINDOW_FRAMES - WINDOW_OVERLAP_FRAMES
PIXELSIZE_MICRONS = float(config.PIXELSIZE_MICRONS)

RUN_LOAD_DATA = True
RUN_EXPORTS = True

BEHAVIOR_STATE_ORDER = ["bound", "confined", "transport", "exploratory"]
COLORBLIND_HEX = ["#0173B2", "#DE8F05", "#029E73", "#CC78BC", "#CA9161", "#56B4E9", "#ECE133", "#D55E00"]

print("Window predictions:", TRANSFORMER_WINDOWED_PARQUET)
print("Results:", RESULTS_DIR)
print("Timing:", {
    "dt_s": FRAME_INTERVAL,
    "time_window_frames": TIME_WINDOW_FRAMES,
    "overlap_frames": WINDOW_OVERLAP_FRAMES,
    "step_frames": WINDOW_STEP_FRAMES,
})

In [ ]:
# =============================================================================
# 1. Load predictions
# =============================================================================
if RUN_LOAD_DATA:
    predicted_windowed_df = pl.read_parquet(TRANSFORMER_WINDOWED_PARQUET)
    predicted_instant_df = pl.read_parquet(TRANSFORMER_INSTANT_PARQUET)
else:
    raise RuntimeError("Set RUN_LOAD_DATA=True")

print("predicted_windowed_df:", predicted_windowed_df.shape)
print("predicted_instant_df:", predicted_instant_df.shape)
display(predicted_windowed_df.head(3))

In [ ]:
# =============================================================================
# 2. Manual filename exclusions
# =============================================================================
RUN_MANUAL_FILENAME_FILTER = True

FILENAMES_TO_FILTER_OUT = [
    "loc-GC_type-vent_mol-HTT_geno-56H20S_013_cropped_tracked",
    "loc-GC_type-vent_77H20S_024_cropped_tracked",
    "loc-GC_type-vent_77H20S_031_cropped_tracked",
    "loc-GC_type-vent_mol-HTT_150H20S_017_cropped_tracked",
    "loc-GC_type-vent_mol-HTT_150H20S_019_cropped_tracked",
    "loc-GC_type-vent_mol-HTT_geno-20H56S_008_cropped_tracked",
    "loc-GC_type-vent_mol-HTT_geno-20H56S_012_cropped_tracked",
    "loc-GC_type-cort_77H20S_031_cropped_tracked",
    "loc-GC_type-cort_77H20S_028_cropped_tracked",
    "loc-GC_type-cort_77H20S_026_cropped_tracked",
    "loc-GC_type-cort_20H77S_045_cropped_tracked",
    "loc-GC_type-cort_20H77S_030_cropped_tracked",
    "loc-GC_type-cort_20H77S_036_cropped_tracked",
    "loc-GC_type-cort_mol-HTT_geno-20H56S_036_cropped_tracked",
    "loc-GC_type-cort_mol-HTT_geno-20H56S_024_cropped_tracked",
    "loc-GC_type-cort_mol-HTT_geno-20H56S_013_cropped_tracked",
    "loc-ES_type-ES_77H20S_015_cropped_tracked",
    "loc-ES_type-ES_20H77S_009_cropped_tracked",
]

if RUN_MANUAL_FILENAME_FILTER:
    before_w, before_i = predicted_windowed_df.shape, predicted_instant_df.shape
    predicted_windowed_df = plasticity.filter_out_filenames(predicted_windowed_df, FILENAMES_TO_FILTER_OUT)
    predicted_instant_df = plasticity.filter_out_filenames(predicted_instant_df, FILENAMES_TO_FILTER_OUT)
    print(f"Excluded {len(FILENAMES_TO_FILTER_OUT)} filenames")
    print("windowed:", before_w, "->", predicted_windowed_df.shape)
    print("instant:", before_i, "->", predicted_instant_df.shape)
else:
    print("Manual filename filter skipped")

In [ ]:
# =============================================================================
# 3. Superwindow MODE: set1 multi-length (default) OR fixed L=15 (NO_DTW)
# =============================================================================
# "multi_length"  -> set1 default (all / even lengths, amalgamated later)
# "fixed_length"  -> original NO_DTW path (single L; default 15)



# SUPERWINDOW_MODE = "fixed_length"  # <-- flip to "fixed_length" for the NO_DTW option

# FIXED_SUPERWINDOW_LENGTH = 4  # used when SUPERWINDOW_MODE == "fixed_length"



SUPERWINDOW_MODE = "multi_length"  # <-- flip to "fixed_length" for the NO_DTW option

FIXED_SUPERWINDOW_LENGTH = 20  # used when SUPERWINDOW_MODE == "fixed_length"

# Set1 = all lengths. Faster alternative: list(range(4, 31, 2))
# MULTI_SUPERWINDOW_LENGTHS = [10, 20]#list(range(4, 31))
# MULTI_SUPERWINDOW_LENGTHS = [2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]
MULTI_SUPERWINDOW_LENGTHS = list(range(5, 30, 5))

if SUPERWINDOW_MODE == "fixed_length":
    SUPERWINDOW_LENGTHS = [int(FIXED_SUPERWINDOW_LENGTH)]
elif SUPERWINDOW_MODE == "multi_length":
    SUPERWINDOW_LENGTHS = list(MULTI_SUPERWINDOW_LENGTHS)
else:
    raise ValueError(f"Unknown SUPERWINDOW_MODE={SUPERWINDOW_MODE!r}")

DISPLAY_SUPERWINDOW_LENGTH = (
    int(FIXED_SUPERWINDOW_LENGTH)
    if SUPERWINDOW_MODE == "fixed_length"
    else int(SUPERWINDOW_LENGTHS[len(SUPERWINDOW_LENGTHS) // 2])
)

SUPERWINDOW_WINDOW_UID_COL = "window_uid"
SUPERWINDOW_INCLUDE_INTERMEDIATE_CARRY_FORWARD = True
SUPERWINDOW_ALLOWED_SINGLE_STATES = ["bound", "exploratory", "confined", "transport"]
SUPERWINDOW_RESOLVED_STATE_COL = "superwindow_resolved_state"
SUPERWINDOW_SOURCE_TAG = "prefilter_carry_forward"
REBUILD_SUPERWINDOWS = False

build_info = plasticity.prepare_superwindow_build_source(
    predicted_windowed_df,
    window_uid_col=SUPERWINDOW_WINDOW_UID_COL,
    include_intermediate_carry_forward=SUPERWINDOW_INCLUDE_INTERMEDIATE_CARRY_FORWARD,
    allowed_states=SUPERWINDOW_ALLOWED_SINGLE_STATES,
    resolved_state_col=SUPERWINDOW_RESOLVED_STATE_COL,
)
SUPERWINDOW_BUILD_STATE_COL = build_info["build_state_col"]
SUPERWINDOW_TIME_COL = build_info["time_col"]
SUPERWINDOW_BUILD_TRACK_COL = build_info["track_col"]
SUPERWINDOW_METADATA_COLS = build_info["metadata_cols"]

print("MODE:", SUPERWINDOW_MODE)
print("STATE:", SUPERWINDOW_BUILD_STATE_COL, "| TRACK:", SUPERWINDOW_BUILD_TRACK_COL, "| TIME:", SUPERWINDOW_TIME_COL)
print("SUPERWINDOW_LENGTHS:", SUPERWINDOW_LENGTHS)
print("DISPLAY_SUPERWINDOW_LENGTH:", DISPLAY_SUPERWINDOW_LENGTH)
if build_info["diagnostics"] is not None:
    display(build_info["diagnostics"])

# Fixed-length path uses the same helper as NO_DTW (single parquet named by source tag + L)
if SUPERWINDOW_MODE == "fixed_length":
    source_tag = f"{SUPERWINDOW_SOURCE_TAG}_fixed{int(FIXED_SUPERWINDOW_LENGTH)}"
    save_path = SUPERWINDOW_SAVE_DIR / f"superwindow_df_all_{source_tag}.parquet"
    superwindow_df_all = plasticity.build_or_load_fixed_superwindows(
        build_info["build_df"],
        save_path=save_path,
        superwindow_length=int(FIXED_SUPERWINDOW_LENGTH),
        state_col=SUPERWINDOW_BUILD_STATE_COL,
        unique_id_col=SUPERWINDOW_BUILD_TRACK_COL,
        time_col=SUPERWINDOW_TIME_COL,
        window_uid_col=SUPERWINDOW_WINDOW_UID_COL,
        additional_cols=SUPERWINDOW_METADATA_COLS,
        rebuild=REBUILD_SUPERWINDOWS,
        write=RUN_EXPORTS,
    )
    # Ensure a length column exists for shared downstream code
    if "superwindow_length" not in superwindow_df_all.columns:
        superwindow_df_all = superwindow_df_all.with_columns(
            pl.lit(int(FIXED_SUPERWINDOW_LENGTH)).alias("superwindow_length")
        )
else:
    superwindow_df_all = plasticity.build_or_load_fixed_superwindows_for_lengths(
        build_info["build_df"],
        save_dir=SUPERWINDOW_SAVE_DIR,
        lengths=SUPERWINDOW_LENGTHS,
        state_col=SUPERWINDOW_BUILD_STATE_COL,
        unique_id_col=SUPERWINDOW_BUILD_TRACK_COL,
        time_col=SUPERWINDOW_TIME_COL,
        window_uid_col=SUPERWINDOW_WINDOW_UID_COL,
        additional_cols=SUPERWINDOW_METADATA_COLS,
        rebuild=REBUILD_SUPERWINDOWS,
        write=RUN_EXPORTS,
        source_tag=SUPERWINDOW_SOURCE_TAG,
    )

print("superwindow_df_all:", superwindow_df_all.shape)
if "superwindow_length" in superwindow_df_all.columns:
    display(superwindow_df_all.group_by("superwindow_length").len().sort("superwindow_length"))
display(superwindow_df_all.head(3))

In [ ]:
# =============================================================================
# 4. FILTERS → analysis tables
# =============================================================================
# Which data set: "HTT_neurons" (the paper figure) or "ES_cells".
# This sets the filters here, and the genotype order and control in the next cells.
DATASET = "HTT_neurons"

FILTERS_BY_DATASET = {
    "HTT_neurons": {
        "mol": ["HTT"],
        "type": ["vent"],
        "location": ["GC"],
        "geno": ["20H20S", "56H20S", "77H20S", "150H20S", "20H56S", "20H77S", "20H150S"],
        "group": ["wtkomt", "HTT77", "HTT56", "HTT150"],
        "nocodazole_concentration": ["none"],
    },
    "ES_cells": {
        "type": ["ES"],
        "group": ["wtkomt", "HTT77", "HTT56", "HTT150"],
        "nocodazole_concentration": ["none"],
    },
}
FILTERS = FILTERS_BY_DATASET[DATASET]
EXCLUDE_FILENAMES = []
INCLUDE_FILENAMES = None
REQUIRE_ALL_SUPERWINDOW_WINDOWS_IN_FILTER = True
DROP_UNRESOLVED_STATES = False

analysis = plasticity.prepare_filtered_analysis_tables(
    predicted_windowed_df,
    predicted_instant_df,
    superwindow_df_all,
    filters=FILTERS,
    include_filenames=INCLUDE_FILENAMES,
    exclude_filenames=EXCLUDE_FILENAMES,
    require_all_superwindow_windows=REQUIRE_ALL_SUPERWINDOW_WINDOWS_IN_FILTER,
    drop_unresolved_states=DROP_UNRESOLVED_STATES,
    state_col_for_unresolved=SUPERWINDOW_BUILD_STATE_COL,
)

analysis_windowed_df = analysis["analysis_windowed_df"]
analysis_instant_df = analysis["analysis_instant_df"]
analysis_superwindow_df = analysis["analysis_superwindow_df"]
filtered_windowed_for_visualization = analysis_windowed_df
filtered_instant_for_visualization = analysis_instant_df
filtered_superwindow_df_for_visualization = analysis_superwindow_df

print("analysis_windowed_df:", analysis_windowed_df.shape)
print("analysis_instant_df:", analysis_instant_df.shape)
print("analysis_superwindow_df:", analysis_superwindow_df.shape)
if analysis_superwindow_df.height == 0:
    raise ValueError("Zero superwindows after filtering.")

display(
    plasticity.summarize_analysis_tables(
        analysis_windowed_df, analysis_instant_df, analysis_superwindow_df, group_col="geno"
    )
)
display(analysis_superwindow_df.head(3))

In [ ]:
# =============================================================================
# 5. Locked analysis / figure settings (SET1 defaults)
# =============================================================================
PLOT_GROUP_COL = "geno"

# Full geno lineup for heatmaps / fans / lines (direct + indirect interleaved by dose)
PLOT_GROUP_ORDER = {
    "HTT_neurons": ["20H20S", "56H20S", "20H56S", "77H20S", "20H77S", "150H20S", "20H150S"],
    "ES_cells": ["RUES2", "HTT72CAG", "HTTKO"],
}[DATASET]

# Optional: restrict figure panels to direct-only (NO_DTW-style)
# PLOT_GROUP_ORDER = ["20H20S", "56H20S", "77H20S", "150H20S"]

SUPERWINDOW_STATE_ORDER = ["bound", "confined", "transport", "exploratory"]
_dark2 = plt.colormaps["Dark2"]
SUPERWINDOW_BASE_PALETTE = [
    to_hex(_dark2(i / max(getattr(_dark2, "N", 8) - 1, 1))) for i in range(getattr(_dark2, "N", 8))
]
SUPERWINDOW_STATE_COLOR_MAP = superwindows.category_palette(SUPERWINDOW_STATE_ORDER, base_colors=SUPERWINDOW_BASE_PALETTE)
PLOT_GROUP_COLOR_MAP = superwindows.category_palette(PLOT_GROUP_ORDER, base_colors=COLORBLIND_HEX)

# --- SET1 defaults ---
SW_UNIT_POLICY = "mean_per_superwindow"
MAX_SW_PER_TRACK = 5
UNIT_RANDOM_STATE = 0
LENGTH_AGG_POLICY = "equal_length_mean"
LENGTHS_FOR_AGG = None  # None = all present lengths

TRANSITION_PROBABILITY_MODE = "mean_per_superwindow"
CONTROL_SCOPE = {"HTT_neurons": "20H20S", "ES_cells": "RUES2"}[DATASET]
FOCAL_STATE = "bound"

SWITCH_QUANTITY = "conditional"
FAN_DISPLAY_MODE = "delta_pct_vs_control_stabilized"
FAN_GLOBAL_SCALE_STAT = "mean"
FAN_STABILIZE_EPS = 0.1

Ltag = (
    f"L{SUPERWINDOW_LENGTHS[0]}"
    if len(SUPERWINDOW_LENGTHS) == 1
    else f"L{min(SUPERWINDOW_LENGTHS)}-{max(SUPERWINDOW_LENGTHS)}"
)
TAG = f"final_{SUPERWINDOW_MODE}_{PLOT_GROUP_COL}_{Ltag}_{SW_UNIT_POLICY}_{LENGTH_AGG_POLICY}"

print("MODE:", SUPERWINDOW_MODE)
print("Lengths:", SUPERWINDOW_LENGTHS)
print("SW_UNIT_POLICY:", SW_UNIT_POLICY, "| MAX_SW_PER_TRACK:", MAX_SW_PER_TRACK)
print("LENGTH_AGG_POLICY:", LENGTH_AGG_POLICY, "| LENGTHS_FOR_AGG:", LENGTHS_FOR_AGG)
print("Quantity / display:", SWITCH_QUANTITY, FAN_DISPLAY_MODE, "eps=", FAN_STABILIZE_EPS)
print("Control / focal:", CONTROL_SCOPE, FOCAL_STATE)
print("PLOT_GROUP_ORDER:", PLOT_GROUP_ORDER)
print("TAG:", TAG)
print("Save dir:", PLASTICITY_PLOTS_DIR)

## Estimate transitions (locked settings)

In [ ]:
# =============================================================================
# 6. Build switch + composition tables
# =============================================================================

sw_overall = plasticity.attach_switch_scope(
    filtered_superwindow_df_for_visualization,
    group_col=PLOT_GROUP_COL,
    scope_col="_switch_scope",
)
sw_overall = sw_overall.loc[sw_overall["_switch_scope"].isin(PLOT_GROUP_ORDER)].copy()

CELL_COL = "cell" if "cell" in sw_overall.columns else None
FILE_COL = "filename" if "filename" in sw_overall.columns else None
print("N superwindows in estimate:", len(sw_overall))
print("cell_col:", CELL_COL, "| file_col:", FILE_COL)

ml = plasticity.build_multi_length_switch_tables(
    sw_overall,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    policy=LENGTH_AGG_POLICY,
    mode=TRANSITION_PROBABILITY_MODE,
    lengths=LENGTHS_FOR_AGG,
    track_col="original_track_id",
    unit_policy=SW_UNIT_POLICY,
    cell_col=CELL_COL,
    file_col=FILE_COL,
    max_sw_per_track=MAX_SW_PER_TRACK,
    random_state=UNIT_RANDOM_STATE,
)
per_length_prob_df = ml["per_length_prob_df"]
per_length_comp_df = ml["per_length_comp_df"]
switch_prob_df = ml["switch_prob_df"]
comp_df = ml["comp_df"]

switch_df, quantity_col = plasticity.prepare_switch_quantity_table(
    switch_prob_df,
    comp_df,
    group_col="_switch_scope",
    switch_quantity=SWITCH_QUANTITY,
    state_order=SUPERWINDOW_STATE_ORDER,
)
display_transition_df = plasticity.build_display_transition_df(
    switch_df,
    group_col="_switch_scope",
    group_order=PLOT_GROUP_ORDER,
    state_order=SUPERWINDOW_STATE_ORDER,
    control_scope=CONTROL_SCOPE,
    display_mode=FAN_DISPLAY_MODE,
    probability_col=quantity_col,
    global_scale_stat=FAN_GLOBAL_SCALE_STAT,
    stabilize_eps=FAN_STABILIZE_EPS,
)
display_transition_df_abs = plasticity.build_display_transition_df(
    switch_df,
    group_col="_switch_scope",
    group_order=PLOT_GROUP_ORDER,
    state_order=SUPERWINDOW_STATE_ORDER,
    control_scope=CONTROL_SCOPE,
    display_mode="absolute",
    probability_col=quantity_col,
)

print("unit_policy:", ml["unit_policy"], "| length_agg:", ml["policy"])
if ml.get("unit_info"):
    print("SWs used/input:", ml["unit_info"]["n_superwindows_used"], "/", ml["unit_info"]["n_superwindows_input"])
display(switch_prob_df.head(16))
display(comp_df.head(16))

## FINAL FIG: fans + heatmaps

In [ ]:
# =============================================================================
# 7. Focal fan + heatmaps — MEDIAN across files (matches recent 8c / 8c-fan)
# =============================================================================
# Previous cell-6 set1 mean_per_superwindow does NOT match the 8c boxplot medians.
# This recomputes:
#   per-file equal-L mean of SW P  →  MEDIAN across filenames (per geno)
# Panels: all conditions in PLOT_GROUP_ORDER (supplementary).

UNIT_COL = "filename"
# Supplementary: all conditions. (Trio-only fans are in 8c-fan.)
FAN_GROUP_ORDER = list(PLOT_GROUP_ORDER)

# None = all lengths (same as recent 8c/8c-fan). Or e.g. [20] / [10, 20].
LENGTHS_KEEP = None

SIZE_BOXES_BY_FREQUENCY = True
SHOW_ZERO_OR_TINY_ARROWS = True
SWITCH_VIZ_LINE_COLOR = "k"
SWITCH_VIZ_MIN_ARROW_PROB = 0.01 if SWITCH_QUANTITY == "conditional" else 0.001

sw_fan = plasticity.attach_switch_scope(
    filtered_superwindow_df_for_visualization,
    group_col=PLOT_GROUP_COL,
    scope_col="_switch_scope",
)
sw_fan = sw_fan.loc[sw_fan["_switch_scope"].astype(str).isin(FAN_GROUP_ORDER)].copy()
if LENGTHS_KEEP is not None:
    sw_fan = sw_fan.loc[
        sw_fan["superwindow_length"].astype(int).isin({int(L) for L in LENGTHS_KEEP})
    ].copy()
if UNIT_COL not in sw_fan.columns:
    raise KeyError(f"{UNIT_COL!r} missing — needed for median-across-files estimand")

print("FAN estimand: median across files (equal-L mean of SW P within file)")
print("FAN_GROUP_ORDER:", FAN_GROUP_ORDER)
print("LENGTHS_KEEP:", LENGTHS_KEEP if LENGTHS_KEEP is not None else "all")
print("N SW:", len(sw_fan))
print(sw_fan.groupby("_switch_scope")[UNIT_COL].nunique().reindex(FAN_GROUP_ORDER))

per_sw = plasticity._per_superwindow_transition_rows(
    sw_fan,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
)
meta = sw_fan[["superwindow_id", UNIT_COL]].drop_duplicates("superwindow_id")
per_sw = per_sw.merge(meta, on="superwindow_id", how="left")

per_occ = plasticity._per_sw_occupancy_rows(
    sw_fan,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
    file_col=UNIT_COL,
)

per_file_L = (
    per_sw.groupby(
        ["_switch_scope", UNIT_COL, "superwindow_length", "from_state", "to_state"],
        dropna=False,
    )["transition_probability_per_superwindow"]
    .mean()
    .reset_index(name="p_file_L")
)
per_file = (
    per_file_L.groupby(
        ["_switch_scope", UNIT_COL, "from_state", "to_state"], dropna=False
    )["p_file_L"]
    .mean()
    .reset_index(name="p_file")
)

occ_file_L = (
    per_occ.groupby(
        ["_switch_scope", UNIT_COL, "superwindow_length", "state"], dropna=False
    )["fraction"]
    .mean()
    .reset_index(name="frac_file_L")
)
occ_file = (
    occ_file_L.groupby(["_switch_scope", UNIT_COL, "state"], dropna=False)["frac_file_L"]
    .mean()
    .reset_index(name="frac_file")
)

switch_prob_df = (
    per_file.groupby(["_switch_scope", "from_state", "to_state"], dropna=False)["p_file"]
    .median()
    .reset_index(name="transition_probability")
)
switch_prob_df["transition_pair"] = (
    switch_prob_df["from_state"].astype(str) + " -> " + switch_prob_df["to_state"].astype(str)
)
switch_prob_df["is_self_transition"] = (
    switch_prob_df["from_state"].astype(str) == switch_prob_df["to_state"].astype(str)
)
switch_prob_df["probability_mode"] = "median_across_files(equal_L_mean_SW)"
n_files = per_file.groupby("_switch_scope")[UNIT_COL].nunique()
switch_prob_df["n_superwindows"] = switch_prob_df["_switch_scope"].map(n_files)

comp_df = (
    occ_file.groupby(["_switch_scope", "state"], dropna=False)["frac_file"]
    .median()
    .reset_index(name="fraction")
)

switch_df, quantity_col = plasticity.prepare_switch_quantity_table(
    switch_prob_df,
    comp_df,
    group_col="_switch_scope",
    switch_quantity=SWITCH_QUANTITY,
    state_order=SUPERWINDOW_STATE_ORDER,
)
display_transition_df = plasticity.build_display_transition_df(
    switch_df,
    group_col="_switch_scope",
    group_order=FAN_GROUP_ORDER,
    state_order=SUPERWINDOW_STATE_ORDER,
    control_scope=CONTROL_SCOPE,
    display_mode=FAN_DISPLAY_MODE,
    probability_col=quantity_col,
    global_scale_stat=FAN_GLOBAL_SCALE_STAT,
    stabilize_eps=FAN_STABILIZE_EPS,
)
display_transition_df_abs = plasticity.build_display_transition_df(
    switch_df,
    group_col="_switch_scope",
    group_order=FAN_GROUP_ORDER,
    state_order=SUPERWINDOW_STATE_ORDER,
    control_scope=CONTROL_SCOPE,
    display_mode="absolute",
    probability_col=quantity_col,
)

Ltag = "allL" if LENGTHS_KEEP is None else ("L" + ",".join(str(L) for L in LENGTHS_KEEP))
tag_d = f"fileMEDIAN_{Ltag}_{SWITCH_QUANTITY}_{FAN_DISPLAY_MODE}"
tag_abs = f"fileMEDIAN_{Ltag}_{SWITCH_QUANTITY}_absolute"
if SIZE_BOXES_BY_FREQUENCY:
    tag_d += "_freqboxes"
    tag_abs += "_freqboxes"
qty_label = "P(next | current)" if SWITCH_QUANTITY == "conditional" else "flux"

fig_fan, _ = plasticity.plot_focal_fan_switch_networks(
    switch_df,
    comp_df,
    group_col="_switch_scope",
    group_order=FAN_GROUP_ORDER,
    state_order=SUPERWINDOW_STATE_ORDER,
    state_colors=SUPERWINDOW_STATE_COLOR_MAP,
    control_scope=CONTROL_SCOPE,
    focal_state=FOCAL_STATE,
    display_mode=FAN_DISPLAY_MODE,
    size_boxes_by_frequency=SIZE_BOXES_BY_FREQUENCY,
    show_zero_or_tiny_arrows=SHOW_ZERO_OR_TINY_ARROWS,
    min_arrow_prob=SWITCH_VIZ_MIN_ARROW_PROB,
    line_color=SWITCH_VIZ_LINE_COLOR,
    global_scale_stat=FAN_GLOBAL_SCALE_STAT,
    stabilize_eps=FAN_STABILIZE_EPS,
    probability_col=quantity_col,
    save_path=PLASTICITY_PLOTS_DIR / f"FINAL_fan_{FOCAL_STATE}_{tag_d}.svg",
    title=f"FINAL fan | MEDIAN across files | {FAN_DISPLAY_MODE} | {Ltag}",
)
plt.show()

fig_hm_abs, _ = plasticity.plot_transition_heatmaps_display(
    display_transition_df_abs,
    group_col="_switch_scope",
    group_order=FAN_GROUP_ORDER,
    state_order=SUPERWINDOW_STATE_ORDER,
    control_scope=CONTROL_SCOPE,
    display_mode="absolute",
    absolute_cbar_label=qty_label,
    save_path=PLASTICITY_PLOTS_DIR / f"FINAL_heatmap_ABS_{tag_abs}.svg",
    title=f"FINAL absolute heatmaps | MEDIAN across files | {Ltag}",
)
plt.show()

fig_hm, _ = plasticity.plot_transition_heatmaps_display(
    display_transition_df,
    group_col="_switch_scope",
    group_order=FAN_GROUP_ORDER,
    state_order=SUPERWINDOW_STATE_ORDER,
    control_scope=CONTROL_SCOPE,
    display_mode=FAN_DISPLAY_MODE,
    absolute_cbar_label=qty_label,
    save_path=PLASTICITY_PLOTS_DIR / f"FINAL_heatmap_REL_{tag_d}.svg",
    title=f"FINAL relative heatmaps | MEDIAN across files | {FAN_DISPLAY_MODE} | {Ltag}",
)
plt.show()

print("Sanity (median file-level P — should match 8c box medians):")
for a, b in [("bound", "exploratory"), ("exploratory", "bound"), ("confined", "exploratory")]:
    bits = [f"{a}->{b}"]
    for g in FAN_GROUP_ORDER:
        sub = switch_prob_df.loc[
            (switch_prob_df["_switch_scope"].astype(str) == g)
            & (switch_prob_df["from_state"].astype(str) == a)
            & (switch_prob_df["to_state"].astype(str) == b),
            "transition_probability",
        ]
        bits.append(f"{g}={float(sub.iloc[0]):.4f}" if len(sub) else f"{g}=n/a")
    print(" ", " | ".join(bits))

print("Saved under:", PLASTICITY_PLOTS_DIR)

## FINAL FIG: direct vs indirect bar plots (focal transitions)

In [ ]:
# =============================================================================
# 8. Bar plots — 4 focal transitions × direct vs indirect (categorical x)
# =============================================================================
FOCAL_PAIRS = [
    ("confined", "bound"),
    ("confined", "exploratory"),
    ("bound", "exploratory"),
    ("exploratory", "bound"),
]

# Equal-spaced categories (not numeric "dose")
CAT_ORDER = [20, 56, 77, 150]
CAT_LABELS = ["20", "56", "77", "150"]
DIRECT = {20: "20H20S", 56: "56H20S", 77: "77H20S", 150: "150H20S"}
INDIRECT = {20: "20H20S", 56: "20H56S", 77: "20H77S", 150: "20H150S"}
X = np.arange(len(CAT_ORDER), dtype=float)
BAR_W = 0.36


def _rel_pct(disp: pd.DataFrame, geno: str, a: str, b: str) -> float:
    if geno == CONTROL_SCOPE:
        return 0.0
    sub = disp.loc[
        (disp["_switch_scope"].astype(str) == geno)
        & (disp["from_state"].astype(str) == a)
        & (disp["to_state"].astype(str) == b)
    ]
    if sub.empty:
        return np.nan
    return float(sub["display_value"].iloc[0])


fig, axes = plt.subplots(2, 2, figsize=(9.5, 7.5), sharex=True)
axes = axes.ravel()

for ax, (a, b) in zip(axes, FOCAL_PAIRS):
    y_dir = [_rel_pct(display_transition_df, DIRECT[c], a, b) for c in CAT_ORDER]
    y_ind = [_rel_pct(display_transition_df, INDIRECT[c], a, b) for c in CAT_ORDER]

    ax.axhline(0.0, color="0.55", lw=0.8, ls="--", zorder=0)
    ax.bar(X - BAR_W / 2, y_dir, width=BAR_W, color="#0072B2", label="direct (*H20S)", zorder=2)
    ax.bar(X + BAR_W / 2, y_ind, width=BAR_W, color="#D55E00", label="indirect (20H*)", zorder=2)
    ax.set_xticks(X)
    ax.set_xticklabels(CAT_LABELS)
    ax.set_title(f"{a} → {b}", fontsize=11)
    ax.set_ylabel("Δ% vs 20H20S (stabilized)")
    ax.grid(True, axis="y", alpha=0.25)

    dir_txt = "  ".join(f"{c}:{v:+.1f}%" for c, v in zip(CAT_ORDER, y_dir) if c != 20)
    ind_txt = "  ".join(f"{c}:{v:+.1f}%" for c, v in zip(CAT_ORDER, y_ind) if c != 20)
    print(f"{a}→{b}  direct:   {dir_txt}")
    print(f"{a}→{b}  indirect: {ind_txt}")

axes[0].legend(loc="best", fontsize=8, frameon=False)
for ax in axes[2:]:
    ax.set_xlabel("genotype category")

fig.suptitle(
    f"FINAL bars | {SUPERWINDOW_MODE} | unit={SW_UNIT_POLICY} | "
    f"agg={LENGTH_AGG_POLICY} | L={SUPERWINDOW_LENGTHS if len(SUPERWINDOW_LENGTHS) <= 6 else f'{min(SUPERWINDOW_LENGTHS)}-{max(SUPERWINDOW_LENGTHS)}'}",
    fontsize=10,
    y=1.02,
)
fig.tight_layout()
out_bars = PLASTICITY_PLOTS_DIR / f"FINAL_bars_focal4_{TAG}_{SWITCH_QUANTITY}.svg"
fig.savefig(out_bars, bbox_inches="tight")
plt.show()
print("saved:", out_bars.name)

## FINAL FIG: per-file bars + dots (mean / median summaries)

In [ ]:
# =============================================================================
# 8b. Per-file (cell) dots — mean and median summary versions
# =============================================================================
# Each dot = one filename: mean of that file's per-superwindow P, equal-mean
# across lengths (same spirit as set1 within-file). Y = stabilised % vs the
# control genotype's mean-of-files. Two figures: summary marker = mean or median
# of the file dots within each genotype.

UNIT_COL = "filename"  # or "cell" if you prefer biological cell IDs
FOCAL_PAIRS = [
    ("confined", "bound"),
    ("confined", "exploratory"),
    ("bound", "exploratory"),
    ("exploratory", "bound"),
]
CAT_ORDER = [20, 56, 77, 150]
CAT_LABELS = ["20", "56", "77", "150"]
DIRECT = {20: "20H20S", 56: "56H20S", 77: "77H20S", 150: "150H20S"}
INDIRECT = {20: "20H20S", 56: "20H56S", 77: "20H77S", 150: "20H150S"}
X = np.arange(len(CAT_ORDER), dtype=float)
JITTER = 0.12
DOT_ALPHA = 0.55
EPS = float(FAN_STABILIZE_EPS)

if UNIT_COL not in sw_overall.columns:
    raise KeyError(f"{UNIT_COL!r} not in sw_overall; columns={list(sw_overall.columns)}")

# Per-SW transition matrices
per_sw = plasticity._per_superwindow_transition_rows(
    sw_overall,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
)
meta = (
    sw_overall[["superwindow_id", UNIT_COL]]
    .drop_duplicates("superwindow_id")
)
per_sw = per_sw.merge(meta, on="superwindow_id", how="left")

# Within (geno, unit, length, pair): mean of SWs  →  then equal mean across lengths
per_unit_L = (
    per_sw.groupby(
        ["_switch_scope", UNIT_COL, "superwindow_length", "from_state", "to_state"],
        dropna=False,
    )["transition_probability_per_superwindow"]
    .mean()
    .reset_index(name="p_unit_L")
)
per_unit = (
    per_unit_L.groupby(
        ["_switch_scope", UNIT_COL, "from_state", "to_state"],
        dropna=False,
    )["p_unit_L"]
    .mean()
    .reset_index(name="p_unit")
)

# Control reference = mean of control-file values (per pair)
ctrl = (
    per_unit.loc[per_unit["_switch_scope"].astype(str) == CONTROL_SCOPE]
    .groupby(["from_state", "to_state"], dropna=False)["p_unit"]
    .mean()
    .rename("p_ctrl")
    .reset_index()
)
per_unit = per_unit.merge(ctrl, on=["from_state", "to_state"], how="left")
per_unit["rel_pct"] = 100.0 * (per_unit["p_unit"] - per_unit["p_ctrl"]) / (per_unit["p_ctrl"] + EPS)

print(f"Unit column: {UNIT_COL}")
print("N units by geno:")
print(per_unit.groupby("_switch_scope")[UNIT_COL].nunique().reindex(PLOT_GROUP_ORDER))


def _series_vals(geno: str, a: str, b: str) -> np.ndarray:
    sub = per_unit.loc[
        (per_unit["_switch_scope"].astype(str) == geno)
        & (per_unit["from_state"].astype(str) == a)
        & (per_unit["to_state"].astype(str) == b),
        "rel_pct",
    ]
    return sub.to_numpy(dtype=float)


BAR_W = 0.36


def _plot_dot_panels(summary: str):
    """Bars = mean/median of file values; dots = individual filenames."""
    fig, axes = plt.subplots(2, 2, figsize=(9.5, 7.5), sharex=True)
    axes = axes.ravel()
    rng = np.random.default_rng(UNIT_RANDOM_STATE)

    for ax, (a, b) in zip(axes, FOCAL_PAIRS):
        ax.axhline(0.0, color="0.55", lw=0.8, ls="--", zorder=0)

        for series, genos, color, x_off, label in (
            ("direct", DIRECT, "#0072B2", -BAR_W / 2, "direct (*H20S)"),
            ("indirect", INDIRECT, "#D55E00", +BAR_W / 2, "indirect (20H*)"),
        ):
            sums = []
            for i, cat in enumerate(CAT_ORDER):
                vals = _series_vals(genos[cat], a, b)
                vals = vals[np.isfinite(vals)]
                if vals.size == 0:
                    sums.append(0.0)
                    continue
                sums.append(float(np.mean(vals) if summary == "mean" else np.median(vals)))
            ax.bar(
                X + x_off,
                sums,
                width=BAR_W,
                color=color,
                alpha=0.85,
                zorder=1,
                label=label,
            )
            for i, cat in enumerate(CAT_ORDER):
                vals = _series_vals(genos[cat], a, b)
                vals = vals[np.isfinite(vals)]
                if vals.size == 0:
                    continue
                jj = rng.uniform(-JITTER, JITTER, size=vals.size)
                ax.scatter(
                    np.full(vals.size, X[i] + x_off) + jj,
                    vals,
                    s=18,
                    color="k",
                    alpha=DOT_ALPHA,
                    edgecolors="none",
                    zorder=3,
                )

        ax.set_xticks(X)
        ax.set_xticklabels(CAT_LABELS)
        ax.set_title(f"{a} → {b}", fontsize=11)
        ax.set_ylabel("Δ% vs 20H20S (stabilized)")
        ax.grid(True, axis="y", alpha=0.25)

        for series, genos in (("direct", DIRECT), ("indirect", INDIRECT)):
            bits = []
            for cat in CAT_ORDER:
                if cat == 20:
                    continue
                vals = _series_vals(genos[cat], a, b)
                vals = vals[np.isfinite(vals)]
                if vals.size == 0:
                    bits.append(f"{cat}:n/a")
                else:
                    v = float(np.mean(vals) if summary == "mean" else np.median(vals))
                    bits.append(f"{cat}:{v:+.1f}% (n={vals.size})")
            print(f"  [{summary}] {a}→{b} {series}: " + "  ".join(bits))

    axes[0].legend(loc="best", fontsize=8, frameon=False)
    for ax in axes[2:]:
        ax.set_xlabel("genotype category")

    fig.suptitle(
        f"FINAL per-{UNIT_COL} | bars={summary} of files | dots=files | {SUPERWINDOW_MODE} | "
        f"mean SW → equal L | eps={EPS}",
        fontsize=10,
        y=1.02,
    )
    fig.tight_layout()
    out = PLASTICITY_PLOTS_DIR / f"FINAL_bars+dots_focal4_{summary}_per-{UNIT_COL}_{TAG}_{SWITCH_QUANTITY}.svg"
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    print("saved:", out.name)
    return fig


print("\n=== MEAN summary of file dots ===")
_plot_dot_panels("mean")
print("\n=== MEDIAN summary of file dots ===")
_plot_dot_panels("median")
print("\nDONE.")

## FINAL FIG: fans for 20H20S / 150H20S / 20H150S (**median** across files; matches 8c boxes)

In [ ]:
# =============================================================================
# 8c-fan. Focal fans for 20H20S / 150H20S / 20H150S — MEDIAN across files
# =============================================================================
# Matches the boxplot *median* summary in cell 8c:
#   per-file equal-L mean of SW P  →  MEDIAN across filenames (per geno).
# (Cell 8c bars are means; boxes show medians — this fan follows the medians.)

FOCUS_GENOS = {"HTT_neurons": ["20H20S", "150H20S", "20H150S"], "ES_cells": ["RUES2", "HTT72CAG", "HTTKO"]}[DATASET]
CONTROL = CONTROL_SCOPE
FOCAL = "bound"
UNIT_COL = "filename"

# Match 8c: all lengths (None) or e.g. [20] / [10, 20]
LENGTHS_KEEP = None

FAN_SWITCH_QUANTITY = "conditional"
FAN_DISPLAY_MODE = "delta_pct_vs_control_stabilized"  # or "absolute"
FAN_STABILIZE_EPS = 0.1
FAN_GLOBAL_SCALE_STAT = "mean"
SIZE_BOXES_BY_FREQUENCY = True
SHOW_ZERO_OR_TINY_ARROWS = True
SWITCH_VIZ_LINE_COLOR = "k"
SWITCH_VIZ_MIN_ARROW_PROB = 0.01

if UNIT_COL not in (
    filtered_superwindow_df_for_visualization.columns
    if hasattr(filtered_superwindow_df_for_visualization, "columns")
    else []
) and UNIT_COL not in getattr(sw_overall, "columns", []):
    # sw_overall is the usual source after cell 6
    pass

sw_fan = plasticity.attach_switch_scope(
    filtered_superwindow_df_for_visualization,
    group_col=PLOT_GROUP_COL,
    scope_col="_switch_scope",
)
sw_fan = sw_fan.loc[sw_fan["_switch_scope"].astype(str).isin(FOCUS_GENOS)].copy()
if LENGTHS_KEEP is not None:
    sw_fan = sw_fan.loc[
        sw_fan["superwindow_length"].astype(int).isin({int(L) for L in LENGTHS_KEEP})
    ].copy()
if UNIT_COL not in sw_fan.columns:
    raise KeyError(f"{UNIT_COL!r} missing")

print("N SW:", len(sw_fan))
print("N files by geno:")
print(sw_fan.groupby("_switch_scope")[UNIT_COL].nunique().reindex(FOCUS_GENOS))

# --- per-SW transitions + occupancy ---
per_sw = plasticity._per_superwindow_transition_rows(
    sw_fan,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
)
meta = sw_fan[["superwindow_id", UNIT_COL]].drop_duplicates("superwindow_id")
per_sw = per_sw.merge(meta, on="superwindow_id", how="left")

per_occ = plasticity._per_sw_occupancy_rows(
    sw_fan,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
    file_col=UNIT_COL,
)

# (geno, file, L, pair) mean over SWs → equal mean across L → file-level P
per_file_L = (
    per_sw.groupby(
        ["_switch_scope", UNIT_COL, "superwindow_length", "from_state", "to_state"],
        dropna=False,
    )["transition_probability_per_superwindow"]
    .mean()
    .reset_index(name="p_file_L")
)
per_file = (
    per_file_L.groupby(
        ["_switch_scope", UNIT_COL, "from_state", "to_state"],
        dropna=False,
    )["p_file_L"]
    .mean()
    .reset_index(name="p_file")
)

# same for composition fractions
occ_file_L = (
    per_occ.groupby(
        ["_switch_scope", UNIT_COL, "superwindow_length", "state"],
        dropna=False,
    )["fraction"]
    .mean()
    .reset_index(name="frac_file_L")
)
occ_file = (
    occ_file_L.groupby(
        ["_switch_scope", UNIT_COL, "state"],
        dropna=False,
    )["frac_file_L"]
    .mean()
    .reset_index(name="frac_file")
)

# MEDIAN across files → geno-level tables
switch_prob_df = (
    per_file.groupby(["_switch_scope", "from_state", "to_state"], dropna=False)["p_file"]
    .median()
    .reset_index(name="transition_probability")
)
switch_prob_df["transition_pair"] = (
    switch_prob_df["from_state"].astype(str) + " -> " + switch_prob_df["to_state"].astype(str)
)
switch_prob_df["is_self_transition"] = (
    switch_prob_df["from_state"].astype(str) == switch_prob_df["to_state"].astype(str)
)
switch_prob_df["probability_mode"] = "median_across_files(equal_L_mean_SW)"
n_files = per_file.groupby("_switch_scope")[UNIT_COL].nunique()
switch_prob_df["n_superwindows"] = switch_prob_df["_switch_scope"].map(n_files)

comp_df = (
    occ_file.groupby(["_switch_scope", "state"], dropna=False)["frac_file"]
    .median()
    .reset_index(name="fraction")
)

switch_df_fan, qty_col_fan = plasticity.prepare_switch_quantity_table(
    switch_prob_df,
    comp_df,
    group_col="_switch_scope",
    switch_quantity=FAN_SWITCH_QUANTITY,
    state_order=SUPERWINDOW_STATE_ORDER,
)

Ltag = "allL" if LENGTHS_KEEP is None else ("L" + ",".join(str(L) for L in LENGTHS_KEEP))
tag_fan = f"fan_20_150_fileMEDIAN_{Ltag}_{FAN_SWITCH_QUANTITY}_{FAN_DISPLAY_MODE}"
if SIZE_BOXES_BY_FREQUENCY:
    tag_fan += "_freqboxes"

fig_fan, _ = plasticity.plot_focal_fan_switch_networks(
    switch_df_fan,
    comp_df,
    group_col="_switch_scope",
    group_order=FOCUS_GENOS,
    state_order=SUPERWINDOW_STATE_ORDER,
    state_colors=SUPERWINDOW_STATE_COLOR_MAP,
    control_scope=CONTROL,
    focal_state=FOCAL,
    display_mode=FAN_DISPLAY_MODE,
    size_boxes_by_frequency=SIZE_BOXES_BY_FREQUENCY,
    show_zero_or_tiny_arrows=SHOW_ZERO_OR_TINY_ARROWS,
    min_arrow_prob=SWITCH_VIZ_MIN_ARROW_PROB,
    line_color=SWITCH_VIZ_LINE_COLOR,
    global_scale_stat=FAN_GLOBAL_SCALE_STAT,
    stabilize_eps=FAN_STABILIZE_EPS,
    probability_col=qty_col_fan,
    save_path=PLASTICITY_PLOTS_DIR / f"FINAL_{tag_fan}.svg",
    title=(
        f"Fan | 20 vs 150 trio | MEDIAN across files (equal-L SW mean) | "
        f"{FAN_DISPLAY_MODE} | {Ltag}"
    ),
)
plt.show()
print("saved:", f"FINAL_{tag_fan}.svg")

if FAN_DISPLAY_MODE != "absolute":
    tag_abs = f"fan_20_150_fileMEDIAN_{Ltag}_{FAN_SWITCH_QUANTITY}_absolute"
    if SIZE_BOXES_BY_FREQUENCY:
        tag_abs += "_freqboxes"
    fig_fan_abs, _ = plasticity.plot_focal_fan_switch_networks(
        switch_df_fan,
        comp_df,
        group_col="_switch_scope",
        group_order=FOCUS_GENOS,
        state_order=SUPERWINDOW_STATE_ORDER,
        state_colors=SUPERWINDOW_STATE_COLOR_MAP,
        control_scope=CONTROL,
        focal_state=FOCAL,
        display_mode="absolute",
        size_boxes_by_frequency=SIZE_BOXES_BY_FREQUENCY,
        show_zero_or_tiny_arrows=SHOW_ZERO_OR_TINY_ARROWS,
        min_arrow_prob=SWITCH_VIZ_MIN_ARROW_PROB,
        line_color=SWITCH_VIZ_LINE_COLOR,
        probability_col=qty_col_fan,
        save_path=PLASTICITY_PLOTS_DIR / f"FINAL_{tag_abs}.svg",
        title=f"Fan ABSOLUTE | MEDIAN across files | {Ltag}",
    )
    plt.show()
    print("saved:", f"FINAL_{tag_abs}.svg")

# Quick sanity: print median P for a few pairs (should track 8c box medians)
print("\nMedian file-level P (should ~ match 8c boxplot medians):")
for a, b in [("bound", "exploratory"), ("exploratory", "bound"), ("confined", "exploratory")]:
    bits = [f"{a}->{b}"]
    for g in FOCUS_GENOS:
        sub = switch_prob_df.loc[
            (switch_prob_df["_switch_scope"].astype(str) == g)
            & (switch_prob_df["from_state"].astype(str) == a)
            & (switch_prob_df["to_state"].astype(str) == b),
            "transition_probability",
        ]
        bits.append(f"{g}={float(sub.iloc[0]):.4f}" if len(sub) else f"{g}=n/a")
    print("  " + " | ".join(bits))

print("DONE 8c-fan (median across files).")

## Bound dwell time — two complementary estimands

**A. Superwindow-local:** run lengths of `bound` inside each SW `state_sequence`. Reports all runs vs *interior* runs (not touching SW edges) to reduce L-censoring bias; same **median across files** summary as the fans.

**B. Whole-track:** concatenate window labels along each track (no SW chopping). Flags runs that hit track start/end; reports complete vs all-visit dwells. Same file-median estimand so A and B are comparable.

In [ ]:
# =============================================================================
# 8c-dwell-A. Bound dwell INSIDE superwindows (run lengths in state_sequence)
# =============================================================================
# Estimand (matches fans / 8c):
#   per-file summary of bound runs → MEDIAN across filenames (per geno).
# Clever bit: also keep INTERIOR runs only (do not touch SW start/end).
# Those are less L-censored — if all-runs and interior-runs disagree a lot,
# SW length is chopping bound episodes.
from sptnano.superwindows import run_lengths_from_sequence

DWELL_STATE = "bound"
DWELL_GENOS = list(FOCUS_GENOS) if "FOCUS_GENOS" in dir() else list(PLOT_GROUP_ORDER)
DWELL_UNIT_COL = "filename"
LENGTHS_KEEP_DWELL = None  # None = all SW lengths present; or e.g. [10, 20]
# seconds per consecutive window step (overlap-aware)
SEC_PER_WINDOW_STEP = float(WINDOW_STEP_FRAMES) * float(FRAME_INTERVAL)

sw_d = plasticity.attach_switch_scope(
    filtered_superwindow_df_for_visualization,
    group_col=PLOT_GROUP_COL,
    scope_col="_switch_scope",
)
sw_d = sw_d.loc[sw_d["_switch_scope"].astype(str).isin(DWELL_GENOS)].copy()
if LENGTHS_KEEP_DWELL is not None:
    sw_d = sw_d.loc[
        sw_d["superwindow_length"].astype(int).isin({int(L) for L in LENGTHS_KEEP_DWELL})
    ].copy()
if DWELL_UNIT_COL not in sw_d.columns:
    raise KeyError(f"{DWELL_UNIT_COL!r} missing on superwindows")
if "state_sequence" not in sw_d.columns:
    raise KeyError("state_sequence missing — rebuild superwindows")

rows = []
for _, r in sw_d.iterrows():
    raw = r["state_sequence"]
    if raw is None:
        continue
    seq = list(raw)
    if len(seq) == 0:
        continue
    L = len(seq)
    geno = str(r["_switch_scope"])
    fn = r[DWELL_UNIT_COL]
    sw_len = int(r["superwindow_length"]) if "superwindow_length" in r.index else L
    sid = r["superwindow_id"] if "superwindow_id" in r.index else None
    pos = 0
    for st, ln in run_lengths_from_sequence(seq):
        if str(st) != DWELL_STATE:
            pos += ln
            continue
        touches_start = pos == 0
        touches_end = (pos + ln) == L
        interior = (not touches_start) and (not touches_end)
        rows.append(
            {
                "_switch_scope": geno,
                DWELL_UNIT_COL: fn,
                "superwindow_id": sid,
                "superwindow_length": sw_len,
                "dwell_windows": int(ln),
                "dwell_s": float(ln) * SEC_PER_WINDOW_STEP,
                "touches_sw_edge": bool(touches_start or touches_end),
                "interior": bool(interior),
                "pure_bound_sw": bool(L == ln),
            }
        )
        pos += ln

dwell_sw_runs = pd.DataFrame(rows)
if dwell_sw_runs.empty:
    raise ValueError(f"No {DWELL_STATE!r} runs found in superwindow sequences")

# per-file: median dwell (windows) for all vs interior runs; also pure-bound SW rate
def _file_sw_summary(g: pd.DataFrame) -> pd.Series:
    all_m = float(g["dwell_windows"].median())
    interior = g.loc[g["interior"]]
    int_m = float(interior["dwell_windows"].median()) if len(interior) else np.nan
    # one row per SW for pure-bound fraction
    sw_pure = g.groupby("superwindow_id")["pure_bound_sw"].max()
    return pd.Series(
        {
            "n_bound_runs": int(len(g)),
            "n_interior_runs": int(len(interior)),
            "median_dwell_windows_all": all_m,
            "median_dwell_windows_interior": int_m,
            "median_dwell_s_all": all_m * SEC_PER_WINDOW_STEP,
            "median_dwell_s_interior": int_m * SEC_PER_WINDOW_STEP if np.isfinite(int_m) else np.nan,
            "frac_pure_bound_sw": float(sw_pure.mean()) if len(sw_pure) else np.nan,
            "n_sw_with_bound": int(sw_pure.shape[0]),
        }
    )


per_file_sw = (
    dwell_sw_runs.groupby(["_switch_scope", DWELL_UNIT_COL], dropna=False)
    .apply(_file_sw_summary)
    .reset_index()
)
# drop leftover group-key columns if pandas duplicated them into the frame
per_file_sw = per_file_sw.loc[:, ~per_file_sw.columns.duplicated()]

# geno-level = MEDIAN across files (fan-compatible)
summary_sw = (
    per_file_sw.groupby("_switch_scope", dropna=False)
    .agg(
        n_files=(DWELL_UNIT_COL, "nunique"),
        median_of_file_median_dwell_windows_all=("median_dwell_windows_all", "median"),
        median_of_file_median_dwell_windows_interior=("median_dwell_windows_interior", "median"),
        median_of_file_median_dwell_s_all=("median_dwell_s_all", "median"),
        median_of_file_median_dwell_s_interior=("median_dwell_s_interior", "median"),
        median_frac_pure_bound_sw=("frac_pure_bound_sw", "median"),
    )
    .reindex(DWELL_GENOS)
    .reset_index()
)

print(
    f"SW-local {DWELL_STATE} dwell | step={WINDOW_STEP_FRAMES} frames "
    f"→ {SEC_PER_WINDOW_STEP:.4f} s/window | LENGTHS_KEEP={LENGTHS_KEEP_DWELL}"
)
print("N bound runs:", len(dwell_sw_runs), "| interior:", int(dwell_sw_runs["interior"].sum()))
display(summary_sw)

# boxplots of per-file medians (all vs interior)
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6), constrained_layout=True)
plot_specs = [
    ("median_dwell_windows_all", "file median dwell (windows)\nall bound runs"),
    ("median_dwell_windows_interior", "file median dwell (windows)\ninterior runs only"),
    ("frac_pure_bound_sw", "file fraction of SWs\nthat are pure-bound"),
]
palette = {g: PLOT_GROUP_COLOR_MAP.get(g, "#888888") for g in DWELL_GENOS}
for ax, (col, title) in zip(axes, plot_specs):
    sub = per_file_sw.dropna(subset=[col])
    sns.boxplot(
        data=sub,
        x="_switch_scope",
        y=col,
        order=DWELL_GENOS,
        hue="_switch_scope",
        palette=palette,
        legend=False,
        ax=ax,
        showfliers=False,
    )
    sns.stripplot(
        data=sub,
        x="_switch_scope",
        y=col,
        order=DWELL_GENOS,
        color="k",
        size=3,
        alpha=0.45,
        ax=ax,
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)

Ltag_d = "allL" if LENGTHS_KEEP_DWELL is None else ("L" + ",".join(map(str, LENGTHS_KEEP_DWELL)))
save_a = PLASTICITY_PLOTS_DIR / f"FINAL_dwell_bound_SUPERWINDOW_fileMEDIAN_{Ltag_d}.svg"
fig.savefig(save_a, bbox_inches="tight")
plt.show()
print("saved:", save_a.name)
print("DONE 8c-dwell-A (superwindow-local).")

In [ ]:
# =============================================================================
# 8c-dwell-A2. Bound ENTER vs EXIT (median + mean across files)
# =============================================================================
# EXIT:  P_exit = 1 - P(bound->bound)  [= SUM of P(to|bound), to!=bound]
# ENTER: P_enter = P(bound | not bound) via occupancy-WEIGHTED AVERAGE of
#        P(bound|from) — do NOT sum those rows.
#
# LENGTH BALANCING (this cell only — flip and re-run):
#   BALANCE_SHORT_VS_LONG = True
#     → within file: mean over SWs at each L, THEN equal mean across L
#       (short and long get equal vote).
#   BALANCE_SHORT_VS_LONG = False
#     → within file: pool all SWs across L (weight ∝ n_SW; short usually wins).
#   per_file / ENTER-EXIT / summaries are rebuilt after this choice.
#
# Across files: both MEDIAN and MEAN of the per-file values.

DWELL_STATE = "bound"
DWELL_GENOS = list(FOCUS_GENOS) if "FOCUS_GENOS" in dir() else list(PLOT_GROUP_ORDER)
DWELL_UNIT_COL = "filename"
LENGTHS_KEEP_IO = None  # None = all; or e.g. [10, 20]

# --- THIS-CELL-ONLY toggle ---
BALANCE_SHORT_VS_LONG = True  # False → pool SWs (short L dominates)

OTHER_STATES = [s for s in SUPERWINDOW_STATE_ORDER if str(s) != DWELL_STATE]

sw_io = plasticity.attach_switch_scope(
    filtered_superwindow_df_for_visualization,
    group_col=PLOT_GROUP_COL,
    scope_col="_switch_scope",
)
sw_io = sw_io.loc[sw_io["_switch_scope"].astype(str).isin(DWELL_GENOS)].copy()
if LENGTHS_KEEP_IO is not None:
    sw_io = sw_io.loc[
        sw_io["superwindow_length"].astype(int).isin({int(L) for L in LENGTHS_KEEP_IO})
    ].copy()
if DWELL_UNIT_COL not in sw_io.columns:
    raise KeyError(f"{DWELL_UNIT_COL!r} missing")

# --- per-SW transitions + occupancy ---
per_sw = plasticity._per_superwindow_transition_rows(
    sw_io,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
)
meta = sw_io[["superwindow_id", DWELL_UNIT_COL]].drop_duplicates("superwindow_id")
per_sw = per_sw.merge(meta, on="superwindow_id", how="left")

per_occ = plasticity._per_sw_occupancy_rows(
    sw_io,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
    file_col=DWELL_UNIT_COL,
)

# Rebuild file-level P / occupancy under the chosen length policy
if BALANCE_SHORT_VS_LONG:
    per_file_L = (
        per_sw.groupby(
            ["_switch_scope", DWELL_UNIT_COL, "superwindow_length", "from_state", "to_state"],
            dropna=False,
        )["transition_probability_per_superwindow"]
        .mean()
        .reset_index(name="p_file_L")
    )
    per_file = (
        per_file_L.groupby(
            ["_switch_scope", DWELL_UNIT_COL, "from_state", "to_state"], dropna=False
        )["p_file_L"]
        .mean()
        .reset_index(name="p_file")
    )
    occ_file_L = (
        per_occ.groupby(
            ["_switch_scope", DWELL_UNIT_COL, "superwindow_length", "state"], dropna=False
        )["fraction"]
        .mean()
        .reset_index(name="frac_file_L")
    )
    occ_file = (
        occ_file_L.groupby(["_switch_scope", DWELL_UNIT_COL, "state"], dropna=False)["frac_file_L"]
        .mean()
        .reset_index(name="frac_file")
    )
    length_policy_tag = "equalL"
else:
    per_file = (
        per_sw.groupby(
            ["_switch_scope", DWELL_UNIT_COL, "from_state", "to_state"], dropna=False
        )["transition_probability_per_superwindow"]
        .mean()
        .reset_index(name="p_file")
    )
    occ_file = (
        per_occ.groupby(["_switch_scope", DWELL_UNIT_COL, "state"], dropna=False)["fraction"]
        .mean()
        .reset_index(name="frac_file")
    )
    length_policy_tag = "pooledSW"

_n_by_L = (
    sw_io.groupby(["_switch_scope", "superwindow_length"], dropna=False)
    .size()
    .rename("n_sw")
    .reset_index()
)
print(f"BALANCE_SHORT_VS_LONG={BALANCE_SHORT_VS_LONG} → file P via '{length_policy_tag}'")
print("N superwindows by geno × L (pooling weights by these counts when False):")
display(
    _n_by_L.pivot_table(
        index="superwindow_length",
        columns="_switch_scope",
        values="n_sw",
        aggfunc="sum",
    ).reindex(columns=DWELL_GENOS)
)

# --- per-file ENTER / EXIT (rebuilt from balanced/unbalanced per_file) ---
io_rows = []
partner_rows = []
for (geno, fn), gP in per_file.groupby(["_switch_scope", DWELL_UNIT_COL], sort=False):
    gO = occ_file.loc[
        (occ_file["_switch_scope"] == geno) & (occ_file[DWELL_UNIT_COL] == fn)
    ]
    occ = {str(s): float(v) for s, v in zip(gO["state"].astype(str), gO["frac_file"])}

    def _p(frm, to, _gP=gP):
        sub = _gP.loc[
            (_gP["from_state"].astype(str) == str(frm))
            & (_gP["to_state"].astype(str) == str(to)),
            "p_file",
        ]
        return float(sub.iloc[0]) if len(sub) else 0.0

    p_stay = _p(DWELL_STATE, DWELL_STATE)
    p_exit = 1.0 - p_stay

    num = 0.0
    den = 0.0
    for frm in OTHER_STATES:
        pi = occ.get(str(frm), 0.0)
        pb = _p(frm, DWELL_STATE)
        num += pi * pb
        den += pi
        partner_rows.append(
            {
                "_switch_scope": geno,
                DWELL_UNIT_COL: fn,
                "direction": "enter",
                "partner": str(frm),
                "p": pb,
                "occupancy_partner": pi,
            }
        )
    p_enter = (num / den) if den > 0 else np.nan

    for to in OTHER_STATES:
        partner_rows.append(
            {
                "_switch_scope": geno,
                DWELL_UNIT_COL: fn,
                "direction": "exit",
                "partner": str(to),
                "p": _p(DWELL_STATE, to),
                "occupancy_partner": occ.get(DWELL_STATE, np.nan),
            }
        )

    io_rows.append(
        {
            "_switch_scope": geno,
            DWELL_UNIT_COL: fn,
            "P_exit_bound": p_exit,
            "P_enter_bound": p_enter,
            "P_stay_bound": p_stay,
            "bound_occupancy": occ.get(DWELL_STATE, np.nan),
            "nonbound_occupancy": den,
        }
    )

per_file_io = pd.DataFrame(io_rows)
per_file_partners = pd.DataFrame(partner_rows)

summary_io_median = (
    per_file_io.groupby("_switch_scope", dropna=False)
    .agg(
        n_files=(DWELL_UNIT_COL, "nunique"),
        median_P_enter_bound=("P_enter_bound", "median"),
        median_P_exit_bound=("P_exit_bound", "median"),
        median_P_stay_bound=("P_stay_bound", "median"),
        median_bound_occupancy=("bound_occupancy", "median"),
    )
    .reindex(DWELL_GENOS)
    .reset_index()
)
summary_io_mean = (
    per_file_io.groupby("_switch_scope", dropna=False)
    .agg(
        n_files=(DWELL_UNIT_COL, "nunique"),
        mean_P_enter_bound=("P_enter_bound", "mean"),
        mean_P_exit_bound=("P_exit_bound", "mean"),
        mean_P_stay_bound=("P_stay_bound", "mean"),
        mean_bound_occupancy=("bound_occupancy", "mean"),
    )
    .reindex(DWELL_GENOS)
    .reset_index()
)
summary_io = summary_io_median  # alias for downstream cells

print(
    "ENTER = occupancy-weighted avg of P(bound|from) over from!=bound "
    "(= P(next=bound | current!=bound))."
)
print("EXIT  = 1 - P(bound|bound)  [= sum of P(to|bound) for to!=bound].")
print(f"Length policy: {length_policy_tag} | then across files → MEDIAN and MEAN")
print("--- MEDIAN across files ---")
display(summary_io_median)
print("--- MEAN across files ---")
display(summary_io_mean)

palette = {g: PLOT_GROUP_COLOR_MAP.get(g, "#888888") for g in DWELL_GENOS}
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.7), constrained_layout=True)

sns.boxplot(
    data=per_file_io,
    x="_switch_scope",
    y="P_enter_bound",
    order=DWELL_GENOS,
    hue="_switch_scope",
    palette=palette,
    legend=False,
    ax=axes[0],
    showfliers=False,
)
sns.stripplot(
    data=per_file_io,
    x="_switch_scope",
    y="P_enter_bound",
    order=DWELL_GENOS,
    color="k",
    size=3,
    alpha=0.45,
    ax=axes[0],
)
axes[0].set_title(f"ENTER bound [{length_policy_tag}]\nP(next=bound | current!=bound)")
axes[0].set_xlabel("")
axes[0].set_ylabel("probability")
axes[0].set_ylim(0, None)
axes[0].tick_params(axis="x", rotation=25)

sns.boxplot(
    data=per_file_io,
    x="_switch_scope",
    y="P_exit_bound",
    order=DWELL_GENOS,
    hue="_switch_scope",
    palette=palette,
    legend=False,
    ax=axes[1],
    showfliers=False,
)
sns.stripplot(
    data=per_file_io,
    x="_switch_scope",
    y="P_exit_bound",
    order=DWELL_GENOS,
    color="k",
    size=3,
    alpha=0.45,
    ax=axes[1],
)
axes[1].set_title(f"EXIT bound [{length_policy_tag}]\nP(next!=bound | current=bound)")
axes[1].set_xlabel("")
axes[1].set_ylabel("probability")
axes[1].set_ylim(0, None)
axes[1].tick_params(axis="x", rotation=25)

partner_med = (
    per_file_partners.groupby(["_switch_scope", "direction", "partner"], dropna=False)["p"]
    .median()
    .reset_index(name="median_p")
)
partner_mean = (
    per_file_partners.groupby(["_switch_scope", "direction", "partner"], dropna=False)["p"]
    .mean()
    .reset_index(name="mean_p")
)

sns.barplot(
    data=partner_med.loc[partner_med["direction"] == "exit"],
    x="partner",
    y="median_p",
    hue="_switch_scope",
    order=OTHER_STATES,
    hue_order=DWELL_GENOS,
    palette=palette,
    ax=axes[2],
)
axes[2].set_title(f"EXIT partners (file-MEDIAN) [{length_policy_tag}]\nP(to | bound)")
axes[2].set_xlabel("to-state")
axes[2].set_ylabel("probability")
axes[2].tick_params(axis="x", rotation=25)
axes[2].legend(title="", fontsize=8, frameon=False)

Ltag_io = "allL" if LENGTHS_KEEP_IO is None else ("L" + ",".join(map(str, LENGTHS_KEEP_IO)))
save_io = (
    PLASTICITY_PLOTS_DIR
    / f"FINAL_bound_ENTER_EXIT_fileMEDIAN_{length_policy_tag}_{Ltag_io}.svg"
)
fig.savefig(save_io, bbox_inches="tight")
plt.show()

# MEAN version of the same three panels
fig_m, axes_m = plt.subplots(1, 3, figsize=(12.0, 3.7), constrained_layout=True)
for ax, col, title in [
    (
        axes_m[0],
        "P_enter_bound",
        f"ENTER bound MEAN [{length_policy_tag}]\nP(next=bound | current!=bound)",
    ),
    (
        axes_m[1],
        "P_exit_bound",
        f"EXIT bound MEAN [{length_policy_tag}]\nP(next!=bound | current=bound)",
    ),
]:
    means = (
        per_file_io.groupby("_switch_scope", dropna=False)[col]
        .mean()
        .reindex(DWELL_GENOS)
        .reset_index()
    )
    sns.barplot(
        data=means,
        x="_switch_scope",
        y=col,
        order=DWELL_GENOS,
        hue="_switch_scope",
        palette=palette,
        legend=False,
        ax=ax,
    )
    sns.stripplot(
        data=per_file_io,
        x="_switch_scope",
        y=col,
        order=DWELL_GENOS,
        color="k",
        size=3,
        alpha=0.45,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("probability")
    ax.set_ylim(0, None)
    ax.tick_params(axis="x", rotation=25)

sns.barplot(
    data=partner_mean.loc[partner_mean["direction"] == "exit"],
    x="partner",
    y="mean_p",
    hue="_switch_scope",
    order=OTHER_STATES,
    hue_order=DWELL_GENOS,
    palette=palette,
    ax=axes_m[2],
)
axes_m[2].set_title(f"EXIT partners (file-MEAN) [{length_policy_tag}]\nP(to | bound)")
axes_m[2].set_xlabel("to-state")
axes_m[2].set_ylabel("probability")
axes_m[2].tick_params(axis="x", rotation=25)
axes_m[2].legend(title="", fontsize=8, frameon=False)

save_io_mean = (
    PLASTICITY_PLOTS_DIR
    / f"FINAL_bound_ENTER_EXIT_fileMEAN_{length_policy_tag}_{Ltag_io}.svg"
)
fig_m.savefig(save_io_mean, bbox_inches="tight")
plt.show()

# companion: ENTER partners — median + mean
fig2, axes2 = plt.subplots(1, 2, figsize=(10.5, 3.4), constrained_layout=True)
sns.barplot(
    data=partner_med.loc[partner_med["direction"] == "enter"],
    x="partner",
    y="median_p",
    hue="_switch_scope",
    order=OTHER_STATES,
    hue_order=DWELL_GENOS,
    palette=palette,
    ax=axes2[0],
)
axes2[0].set_title(f"ENTER partners MEDIAN [{length_policy_tag}]\nP(bound | from)")
axes2[0].set_xlabel("from-state")
axes2[0].set_ylabel("probability")
axes2[0].tick_params(axis="x", rotation=25)
axes2[0].legend(title="", fontsize=8, frameon=False)

sns.barplot(
    data=partner_mean.loc[partner_mean["direction"] == "enter"],
    x="partner",
    y="mean_p",
    hue="_switch_scope",
    order=OTHER_STATES,
    hue_order=DWELL_GENOS,
    palette=palette,
    ax=axes2[1],
)
axes2[1].set_title(f"ENTER partners MEAN [{length_policy_tag}]\nP(bound | from)")
axes2[1].set_xlabel("from-state")
axes2[1].set_ylabel("probability")
axes2[1].tick_params(axis="x", rotation=25)
axes2[1].legend(title="", fontsize=8, frameon=False)

save_io2 = (
    PLASTICITY_PLOTS_DIR
    / f"FINAL_bound_ENTER_partners_fileMEDIAN_MEAN_{length_policy_tag}_{Ltag_io}.svg"
)
fig2.savefig(save_io2, bbox_inches="tight")
plt.show()

print("saved:", save_io.name)
print("saved:", save_io_mean.name)
print("saved:", save_io2.name)
print("DONE 8c-dwell-A2 (enter / exit; balance toggle + median/mean).")

In [ ]:
# =============================================================================
# 8c-dwell-A2W. Bound ENTER/EXIT — SUPERWINDOW-INDEPENDENT (every window switch)
# =============================================================================
# Unlike A2 (per-SW transition matrices → aggregate), this counts EVERY adjacent
# window→window switch along whole tracks in analysis_windowed_df.
#
# Per file:
#   C[from,to] = raw transition counts across all tracks in that file
#   P(to|from) = C[from,to] / sum_to C[from,.]   (row-normalize; empty rows → nan)
#
# ENTER / EXIT (same definitions as A2, from these count-based P):
#   P_exit  = 1 - P(bound|bound)                    # SUM outbound from bound
#   P_enter = sum_{f!=bound} pi_f P(bound|f) / sum pi_f  # occupancy-WEIGHTED AVG
#   pi_f    = fraction of windows in state f within the file
#
# Per-condition balancing (this cell only):
#   BALANCE_FILES_WITHIN_CONDITION = True (default)
#     → each filename gets one vote; geno summary = MEDIAN and MEAN across files
#   BALANCE_FILES_WITHIN_CONDITION = False
#     → pool raw counts across all files in the geno, then one P / ENTER / EXIT
#       (files with more transitions dominate). Median/mean tables are identical.
#
# Same plot families as A2: median panels, mean panels, enter partners.

DWELL_STATE = "bound"
DWELL_GENOS = list(FOCUS_GENOS) if "FOCUS_GENOS" in dir() else list(PLOT_GROUP_ORDER)
DWELL_UNIT_COL = "filename"
OTHER_STATES = [s for s in SUPERWINDOW_STATE_ORDER if str(s) != DWELL_STATE]
ALL_STATES = list(SUPERWINDOW_STATE_ORDER)

# --- THIS-CELL-ONLY toggle ---
BALANCE_FILES_WITHIN_CONDITION = True  # False → pool counts across files in geno

_state_candidates = [
    c
    for c in [
        SUPERWINDOW_BUILD_STATE_COL if "SUPERWINDOW_BUILD_STATE_COL" in dir() else None,
        "superwindow_resolved_state",
        "transformer_behavior_state",
        "transformer_behaviour_state",
        "final_population",
    ]
    if c
]
_track_candidates = [
    c
    for c in [
        SUPERWINDOW_BUILD_TRACK_COL if "SUPERWINDOW_BUILD_TRACK_COL" in dir() else None,
        "unique_id",
        "original_track_id",
        "track_id",
    ]
    if c
]
_time_candidates = [
    c
    for c in [
        SUPERWINDOW_TIME_COL if "SUPERWINDOW_TIME_COL" in dir() else None,
        "time_window",
        "window_number",
        "window_index",
    ]
    if c
]

wdf = analysis_windowed_df
wpdf = wdf.to_pandas() if isinstance(wdf, pl.DataFrame) else wdf.copy()
STATE_COL = next(c for c in _state_candidates if c in wpdf.columns)
TRACK_COL = next(c for c in _track_candidates if c in wpdf.columns)
TIME_COL = next(c for c in _time_candidates if c in wpdf.columns)
for req in [PLOT_GROUP_COL, DWELL_UNIT_COL]:
    if req not in wpdf.columns:
        raise KeyError(f"{req!r} missing on windowed df")

wpdf = wpdf.loc[wpdf[PLOT_GROUP_COL].astype(str).isin(DWELL_GENOS)].copy()
wpdf[STATE_COL] = wpdf[STATE_COL].astype(str)
wpdf = wpdf.sort_values([DWELL_UNIT_COL, TRACK_COL, TIME_COL])

print(
    "SW-independent ENTER/EXIT | every adjacent window switch |",
    {"state": STATE_COL, "track": TRACK_COL, "time": TIME_COL},
)
print(
    "BALANCE_FILES_WITHIN_CONDITION=",
    BALANCE_FILES_WITHIN_CONDITION,
    "| N windows:",
    len(wpdf),
    "| N tracks:",
    wpdf[TRACK_COL].nunique(),
)

STATE_IDX = {str(s): i for i, s in enumerate(ALL_STATES)}
K = len(ALL_STATES)


def _count_transitions(sub: pd.DataFrame) -> np.ndarray:
    C = np.zeros((K, K), dtype=np.int64)
    for _, g in sub.groupby(TRACK_COL, sort=False):
        seq = g[STATE_COL].tolist()
        for a, b in zip(seq[:-1], seq[1:]):
            if a in STATE_IDX and b in STATE_IDX:
                C[STATE_IDX[a], STATE_IDX[b]] += 1
    return C


def _row_normalize(C: np.ndarray) -> np.ndarray:
    P = np.zeros_like(C, dtype=float)
    rs = C.sum(axis=1, keepdims=True)
    np.divide(C, rs, out=P, where=rs > 0)
    return P


def _occupancy(sub: pd.DataFrame) -> dict:
    vc = sub[STATE_COL].value_counts(normalize=True)
    return {str(s): float(vc.get(s, 0.0)) for s in ALL_STATES}


def _metrics_from_subset(sub: pd.DataFrame, scope: str, unit: str) -> tuple[dict, list[dict]]:
    C = _count_transitions(sub)
    P = _row_normalize(C)
    occ = _occupancy(sub)
    ib = STATE_IDX[DWELL_STATE]
    p_stay = float(P[ib, ib]) if C[ib].sum() > 0 else np.nan
    p_exit = (1.0 - p_stay) if np.isfinite(p_stay) else np.nan
    partners = []
    num = den = 0.0
    for s in OTHER_STATES:
        i = STATE_IDX[str(s)]
        pi = float(occ.get(str(s), 0.0))
        pb = float(P[i, ib]) if C[i].sum() > 0 else np.nan
        if np.isfinite(pb):
            num += pi * pb
            den += pi
        partners.append(
            {
                "_switch_scope": scope,
                DWELL_UNIT_COL: unit,
                "direction": "enter",
                "partner": str(s),
                "p": pb,
                "occupancy_partner": pi,
                "n_from_transitions": int(C[i].sum()),
            }
        )
    p_enter = (num / den) if den > 0 else np.nan
    for s in OTHER_STATES:
        i = STATE_IDX[str(s)]
        partners.append(
            {
                "_switch_scope": scope,
                DWELL_UNIT_COL: unit,
                "direction": "exit",
                "partner": str(s),
                "p": float(P[ib, i]) if C[ib].sum() > 0 else np.nan,
                "occupancy_partner": float(occ.get(DWELL_STATE, np.nan)),
                "n_from_transitions": int(C[ib].sum()),
            }
        )
    row = {
        "_switch_scope": scope,
        DWELL_UNIT_COL: unit,
        "P_exit_bound": p_exit,
        "P_enter_bound": p_enter,
        "P_stay_bound": p_stay,
        "bound_occupancy": float(occ.get(DWELL_STATE, np.nan)),
        "nonbound_occupancy": den,
        "n_transitions": int(C.sum()),
        "n_windows": int(len(sub)),
        "n_tracks": int(sub[TRACK_COL].nunique()),
    }
    return row, partners


file_rows, partner_rows = [], []
for (geno, fn), sub in wpdf.groupby([PLOT_GROUP_COL, DWELL_UNIT_COL], sort=False):
    row, partners = _metrics_from_subset(sub, str(geno), fn)
    file_rows.append(row)
    partner_rows.extend(partners)

per_file_io_w = pd.DataFrame(file_rows)
per_file_partners_w = pd.DataFrame(partner_rows)
nfiles_map = per_file_io_w.groupby("_switch_scope")[DWELL_UNIT_COL].nunique()

if BALANCE_FILES_WITHIN_CONDITION:
    balance_tag = "equalFile"
    plot_io = per_file_io_w
    summary_w_median = (
        per_file_io_w.groupby("_switch_scope", dropna=False)
        .agg(
            n_files=(DWELL_UNIT_COL, "nunique"),
            median_P_enter_bound=("P_enter_bound", "median"),
            median_P_exit_bound=("P_exit_bound", "median"),
            median_P_stay_bound=("P_stay_bound", "median"),
            median_bound_occupancy=("bound_occupancy", "median"),
            total_transitions=("n_transitions", "sum"),
        )
        .reindex(DWELL_GENOS)
        .reset_index()
    )
    summary_w_mean = (
        per_file_io_w.groupby("_switch_scope", dropna=False)
        .agg(
            n_files=(DWELL_UNIT_COL, "nunique"),
            mean_P_enter_bound=("P_enter_bound", "mean"),
            mean_P_exit_bound=("P_exit_bound", "mean"),
            mean_P_stay_bound=("P_stay_bound", "mean"),
            mean_bound_occupancy=("bound_occupancy", "mean"),
            total_transitions=("n_transitions", "sum"),
        )
        .reindex(DWELL_GENOS)
        .reset_index()
    )
    partner_med_w = (
        per_file_partners_w.groupby(["_switch_scope", "direction", "partner"], dropna=False)["p"]
        .median()
        .reset_index(name="median_p")
    )
    partner_mean_w = (
        per_file_partners_w.groupby(["_switch_scope", "direction", "partner"], dropna=False)["p"]
        .mean()
        .reset_index(name="mean_p")
    )
else:
    balance_tag = "pooledCounts"
    pooled_rows, pooled_partners = [], []
    for geno in DWELL_GENOS:
        sub = wpdf.loc[wpdf[PLOT_GROUP_COL].astype(str) == geno]
        if sub.empty:
            continue
        row, partners = _metrics_from_subset(sub, geno, "__pooled__")
        pooled_rows.append(row)
        for p in partners:
            pooled_partners.append(
                {
                    "_switch_scope": geno,
                    "direction": p["direction"],
                    "partner": p["partner"],
                    "median_p": p["p"],
                    "mean_p": p["p"],
                }
            )
    plot_io = pd.DataFrame(pooled_rows)
    summary_w_median = plot_io.rename(
        columns={
            "P_enter_bound": "median_P_enter_bound",
            "P_exit_bound": "median_P_exit_bound",
            "P_stay_bound": "median_P_stay_bound",
            "bound_occupancy": "median_bound_occupancy",
            "n_transitions": "total_transitions",
        }
    )[
        [
            "_switch_scope",
            "median_P_enter_bound",
            "median_P_exit_bound",
            "median_P_stay_bound",
            "median_bound_occupancy",
            "total_transitions",
        ]
    ].copy()
    summary_w_median.insert(1, "n_files", summary_w_median["_switch_scope"].map(nfiles_map))
    summary_w_median = summary_w_median.set_index("_switch_scope").reindex(DWELL_GENOS).reset_index()
    summary_w_mean = summary_w_median.rename(
        columns={
            "median_P_enter_bound": "mean_P_enter_bound",
            "median_P_exit_bound": "mean_P_exit_bound",
            "median_P_stay_bound": "mean_P_stay_bound",
            "median_bound_occupancy": "mean_bound_occupancy",
        }
    )
    partner_med_w = pd.DataFrame(pooled_partners)
    partner_mean_w = partner_med_w.copy()

print(f"Balance: {balance_tag} | unit = every adjacent window switch (no superwindows)")
print("--- MEDIAN across files (or pooled point) ---")
display(summary_w_median)
print("--- MEAN across files (or pooled point) ---")
display(summary_w_mean)

palette = {g: PLOT_GROUP_COLOR_MAP.get(g, "#888888") for g in DWELL_GENOS}


def _plot_enter_exit(axes, mode: str):
    """mode: 'box' (per-file dist) or 'meanbar' (mean bars + dots) or 'poolbar'."""
    specs = [
        (0, "P_enter_bound", f"ENTER [{balance_tag}/SW-indep]\nP(next=bound | current!=bound)"),
        (1, "P_exit_bound", f"EXIT [{balance_tag}/SW-indep]\nP(next!=bound | current=bound)"),
    ]
    for ax_i, col, title in specs:
        ax = axes[ax_i]
        if mode == "box":
            sns.boxplot(
                data=plot_io, x="_switch_scope", y=col, order=DWELL_GENOS,
                hue="_switch_scope", palette=palette, legend=False, ax=ax, showfliers=False,
            )
            sns.stripplot(
                data=plot_io, x="_switch_scope", y=col, order=DWELL_GENOS,
                color="k", size=3, alpha=0.45, ax=ax,
            )
        elif mode == "meanbar":
            means = (
                plot_io.groupby("_switch_scope", dropna=False)[col]
                .mean().reindex(DWELL_GENOS).reset_index()
            )
            sns.barplot(
                data=means, x="_switch_scope", y=col, order=DWELL_GENOS,
                hue="_switch_scope", palette=palette, legend=False, ax=ax,
            )
            sns.stripplot(
                data=plot_io, x="_switch_scope", y=col, order=DWELL_GENOS,
                color="k", size=3, alpha=0.45, ax=ax,
            )
        else:
            sns.barplot(
                data=plot_io, x="_switch_scope", y=col, order=DWELL_GENOS,
                hue="_switch_scope", palette=palette, legend=False, ax=ax,
            )
        ax.set_title(title if mode != "meanbar" else title.replace("ENTER", "ENTER MEAN").replace("EXIT", "EXIT MEAN"))
        if mode == "meanbar":
            ax.set_title(title.replace("[", "MEAN ["))
        ax.set_xlabel("")
        ax.set_ylabel("probability")
        ax.set_ylim(0, None)
        ax.tick_params(axis="x", rotation=25)


# MEDIAN / distribution figure
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.7), constrained_layout=True)
_plot_enter_exit(axes, "box" if BALANCE_FILES_WITHIN_CONDITION else "poolbar")
sns.barplot(
    data=partner_med_w.loc[partner_med_w["direction"] == "exit"],
    x="partner", y="median_p", hue="_switch_scope",
    order=OTHER_STATES, hue_order=DWELL_GENOS, palette=palette, ax=axes[2],
)
axes[2].set_title(f"EXIT partners MEDIAN [{balance_tag}]\nP(to | bound)")
axes[2].set_xlabel("to-state")
axes[2].set_ylabel("probability")
axes[2].tick_params(axis="x", rotation=25)
axes[2].legend(title="", fontsize=8, frameon=False)
save_w = PLASTICITY_PLOTS_DIR / f"FINAL_bound_ENTER_EXIT_SWindep_fileMEDIAN_{balance_tag}.svg"
fig.savefig(save_w, bbox_inches="tight")
plt.show()

# MEAN figure
fig_m, axes_m = plt.subplots(1, 3, figsize=(12.0, 3.7), constrained_layout=True)
_plot_enter_exit(axes_m, "meanbar" if BALANCE_FILES_WITHIN_CONDITION else "poolbar")
sns.barplot(
    data=partner_mean_w.loc[partner_mean_w["direction"] == "exit"],
    x="partner", y="mean_p", hue="_switch_scope",
    order=OTHER_STATES, hue_order=DWELL_GENOS, palette=palette, ax=axes_m[2],
)
axes_m[2].set_title(f"EXIT partners MEAN [{balance_tag}]\nP(to | bound)")
axes_m[2].set_xlabel("to-state")
axes_m[2].set_ylabel("probability")
axes_m[2].tick_params(axis="x", rotation=25)
axes_m[2].legend(title="", fontsize=8, frameon=False)
save_wm = PLASTICITY_PLOTS_DIR / f"FINAL_bound_ENTER_EXIT_SWindep_fileMEAN_{balance_tag}.svg"
fig_m.savefig(save_wm, bbox_inches="tight")
plt.show()

# ENTER partners
fig2, axes2 = plt.subplots(1, 2, figsize=(10.5, 3.4), constrained_layout=True)
sns.barplot(
    data=partner_med_w.loc[partner_med_w["direction"] == "enter"],
    x="partner", y="median_p", hue="_switch_scope",
    order=OTHER_STATES, hue_order=DWELL_GENOS, palette=palette, ax=axes2[0],
)
axes2[0].set_title(f"ENTER partners MEDIAN [{balance_tag}/SW-indep]\nP(bound | from)")
axes2[0].set_xlabel("from-state")
axes2[0].set_ylabel("probability")
axes2[0].tick_params(axis="x", rotation=25)
axes2[0].legend(title="", fontsize=8, frameon=False)
sns.barplot(
    data=partner_mean_w.loc[partner_mean_w["direction"] == "enter"],
    x="partner", y="mean_p", hue="_switch_scope",
    order=OTHER_STATES, hue_order=DWELL_GENOS, palette=palette, ax=axes2[1],
)
axes2[1].set_title(f"ENTER partners MEAN [{balance_tag}/SW-indep]\nP(bound | from)")
axes2[1].set_xlabel("from-state")
axes2[1].set_ylabel("probability")
axes2[1].tick_params(axis="x", rotation=25)
axes2[1].legend(title="", fontsize=8, frameon=False)
save_w2 = PLASTICITY_PLOTS_DIR / f"FINAL_bound_ENTER_partners_SWindep_{balance_tag}.svg"
fig2.savefig(save_w2, bbox_inches="tight")
plt.show()

if "summary_io_median" in dir():
    print("\nCompare A2 (SW matrices) vs A2W (every switch) — median P_exit / P_enter:")
    for geno in DWELL_GENOS:
        def _get(df, col):
            s = df.loc[df["_switch_scope"] == geno, col]
            return float(s.iloc[0]) if len(s) else np.nan
        print(
            f"  {geno}: exit A2={_get(summary_io_median,'median_P_exit_bound'):.4f} "
            f"A2W={_get(summary_w_median,'median_P_exit_bound'):.4f} | "
            f"enter A2={_get(summary_io_median,'median_P_enter_bound'):.4f} "
            f"A2W={_get(summary_w_median,'median_P_enter_bound'):.4f}"
        )

print("saved:", save_w.name)
print("saved:", save_wm.name)
print("saved:", save_w2.name)
print("DONE 8c-dwell-A2W (SW-independent every-switch ENTER/EXIT).")

In [ ]:
# =============================================================================
# 8c-dwell-A3. Bound ↔ each state separately (ENTER / EXIT boxplots)
# =============================================================================
# Requires 8c-dwell-A2 (uses per_file_partners).
# For each partner state s ≠ bound:
#   ENTER: P(bound | s)     — one conditional, no pooling/summing
#   EXIT:  P(s | bound)     — one outbound arrow from the bound row
# Dots = files; box = across-file distribution (same units as A2).
if "per_file_partners" not in dir() or per_file_partners is None or len(per_file_partners) == 0:
    raise RuntimeError("Run 8c-dwell-A2 first (needs per_file_partners).")

DWELL_GENOS = list(FOCUS_GENOS) if "FOCUS_GENOS" in dir() else list(PLOT_GROUP_ORDER)
OTHER_STATES = [s for s in SUPERWINDOW_STATE_ORDER if str(s) != "bound"]
palette = {g: PLOT_GROUP_COLOR_MAP.get(g, "#888888") for g in DWELL_GENOS}

pp = per_file_partners.copy()
pp["partner"] = pp["partner"].astype(str)
pp["_switch_scope"] = pp["_switch_scope"].astype(str)

n_p = len(OTHER_STATES)
fig, axes = plt.subplots(
    2,
    n_p,
    figsize=(3.2 * n_p, 6.2),
    sharey="row",
    constrained_layout=True,
)
if n_p == 1:
    axes = np.asarray(axes).reshape(2, 1)

for j, partner in enumerate(OTHER_STATES):
    for i, (direction, ylab, row_title) in enumerate(
        [
            ("enter", f"P(bound | {partner})", f"ENTER from {partner}"),
            ("exit", f"P({partner} | bound)", f"EXIT to {partner}"),
        ]
    ):
        ax = axes[i, j]
        sub = pp.loc[
            (pp["direction"] == direction) & (pp["partner"] == str(partner))
        ]
        if sub.empty:
            ax.set_title(f"{row_title}\n(no data)")
            continue
        sns.boxplot(
            data=sub,
            x="_switch_scope",
            y="p",
            order=DWELL_GENOS,
            hue="_switch_scope",
            palette=palette,
            legend=False,
            ax=ax,
            showfliers=False,
        )
        sns.stripplot(
            data=sub,
            x="_switch_scope",
            y="p",
            order=DWELL_GENOS,
            color="k",
            size=3,
            alpha=0.45,
            ax=ax,
        )
        ax.set_title(row_title, fontsize=10)
        ax.set_xlabel("")
        ax.set_ylabel(ylab if j == 0 else "")
        ax.set_ylim(0, None)
        ax.tick_params(axis="x", rotation=25)

# citation table: median across files per geno × partner × direction
per_edge_summary = (
    pp.groupby(["direction", "partner", "_switch_scope"], dropna=False)["p"]
    .median()
    .rename("median_across_files")
    .reset_index()
)
per_edge_wide = (
    per_edge_summary.pivot_table(
        index=["direction", "partner"],
        columns="_switch_scope",
        values="median_across_files",
        aggfunc="first",
    )
    .reindex(columns=DWELL_GENOS)
    .reset_index()
)
print("Median across files — each edge separately (not pooled):")
display(per_edge_wide)

Ltag_io = globals().get("Ltag_io", "allL")
save_a3 = PLASTICITY_PLOTS_DIR / f"FINAL_bound_ENTER_EXIT_per_state_fileMEDIAN_{Ltag_io}.svg"
fig.savefig(save_a3, bbox_inches="tight")
plt.show()
print("saved:", save_a3.name)
print("DONE 8c-dwell-A3 (per-state enter/exit).")

In [ ]:
# =============================================================================
# 8c-dwell-B. Bound dwell on WHOLE TRACKS (no superwindow chopping)
# =============================================================================
# Build ordered window labels per track from analysis_windowed_df (same filter
# as the rest of this notebook). Run lengths of bound along the full track.
#
# Clever bits:
#   - flag runs that hit track START or END (right/left censored by tracking)
#   - report COMPLETE runs (neither edge) vs ALL visits
#   - same MEDIAN-across-files estimand as A / fans, so compare A vs B apples-to-apples
#   - optional KM-style survival: P(dwell >= k windows) for bound visits
DWELL_STATE = "bound"
DWELL_GENOS = list(FOCUS_GENOS) if "FOCUS_GENOS" in dir() else list(PLOT_GROUP_ORDER)
DWELL_UNIT_COL = "filename"
SEC_PER_WINDOW_STEP = float(WINDOW_STEP_FRAMES) * float(FRAME_INTERVAL)

# Prefer the same resolved/build state used for superwindows
_state_candidates = [
    c
    for c in [
        SUPERWINDOW_BUILD_STATE_COL if "SUPERWINDOW_BUILD_STATE_COL" in dir() else None,
        "superwindow_resolved_state",
        "transformer_behavior_state",
        "transformer_behaviour_state",
        "final_population",
    ]
    if c
]
_track_candidates = [
    c
    for c in [
        SUPERWINDOW_BUILD_TRACK_COL if "SUPERWINDOW_BUILD_TRACK_COL" in dir() else None,
        "unique_id",
        "original_track_id",
        "track_id",
    ]
    if c
]
_time_candidates = [
    c
    for c in [
        SUPERWINDOW_TIME_COL if "SUPERWINDOW_TIME_COL" in dir() else None,
        "time_window",
        "window_number",
        "window_index",
    ]
    if c
]

wdf = analysis_windowed_df
if isinstance(wdf, pl.DataFrame):
    wpdf = wdf.to_pandas()
else:
    wpdf = wdf.copy()

STATE_COL = next(c for c in _state_candidates if c in wpdf.columns)
TRACK_COL = next(c for c in _track_candidates if c in wpdf.columns)
TIME_COL = next(c for c in _time_candidates if c in wpdf.columns)
if PLOT_GROUP_COL not in wpdf.columns:
    raise KeyError(f"{PLOT_GROUP_COL!r} missing on windowed df")
if DWELL_UNIT_COL not in wpdf.columns:
    raise KeyError(f"{DWELL_UNIT_COL!r} missing on windowed df")

wpdf = wpdf.loc[wpdf[PLOT_GROUP_COL].astype(str).isin(DWELL_GENOS)].copy()
wpdf[STATE_COL] = wpdf[STATE_COL].astype(str)
wpdf = wpdf.sort_values([DWELL_UNIT_COL, TRACK_COL, TIME_COL])

print("Track dwell source:", {"state": STATE_COL, "track": TRACK_COL, "time": TIME_COL})
print("N windows:", len(wpdf), "| N tracks:", wpdf[TRACK_COL].nunique())

from sptnano.superwindows import run_lengths_from_sequence

track_rows = []
for (geno, fn, tid), g in wpdf.groupby([PLOT_GROUP_COL, DWELL_UNIT_COL, TRACK_COL], sort=False):
    seq = g[STATE_COL].tolist()
    if not seq:
        continue
    n = len(seq)
    pos = 0
    for st, ln in run_lengths_from_sequence(seq):
        if str(st) != DWELL_STATE:
            pos += ln
            continue
        left_c = pos == 0
        right_c = (pos + ln) == n
        complete = (not left_c) and (not right_c)
        track_rows.append(
            {
                "_switch_scope": str(geno),
                DWELL_UNIT_COL: fn,
                TRACK_COL: tid,
                "dwell_windows": int(ln),
                "dwell_s": float(ln) * SEC_PER_WINDOW_STEP,
                "left_censored": bool(left_c),
                "right_censored": bool(right_c),
                "complete": bool(complete),
                "track_n_windows": int(n),
                "bound_occupancy": float(sum(1 for s in seq if str(s) == DWELL_STATE) / n),
            }
        )
        pos += ln

dwell_track_runs = pd.DataFrame(track_rows)
if dwell_track_runs.empty:
    raise ValueError(f"No {DWELL_STATE!r} runs on whole tracks")


def _file_track_summary(g: pd.DataFrame) -> pd.Series:
    complete = g.loc[g["complete"]]
    return pd.Series(
        {
            "n_bound_runs": int(len(g)),
            "n_complete_runs": int(len(complete)),
            "median_dwell_windows_all": float(g["dwell_windows"].median()),
            "median_dwell_windows_complete": (
                float(complete["dwell_windows"].median()) if len(complete) else np.nan
            ),
            "median_dwell_s_all": float(g["dwell_s"].median()),
            "median_dwell_s_complete": (
                float(complete["dwell_s"].median()) if len(complete) else np.nan
            ),
            "frac_runs_complete": float(complete.shape[0] / len(g)) if len(g) else np.nan,
            # track-level occupancy: one value per track then median in file
            "median_track_bound_occupancy": float(
                g.drop_duplicates(TRACK_COL)["bound_occupancy"].median()
            ),
        }
    )


per_file_tr = (
    dwell_track_runs.groupby(["_switch_scope", DWELL_UNIT_COL], dropna=False)
    .apply(_file_track_summary)
    .reset_index()
)
per_file_tr = per_file_tr.loc[:, ~per_file_tr.columns.duplicated()]

summary_tr = (
    per_file_tr.groupby("_switch_scope", dropna=False)
    .agg(
        n_files=(DWELL_UNIT_COL, "nunique"),
        median_of_file_median_dwell_windows_all=("median_dwell_windows_all", "median"),
        median_of_file_median_dwell_windows_complete=("median_dwell_windows_complete", "median"),
        median_of_file_median_dwell_s_all=("median_dwell_s_all", "median"),
        median_of_file_median_dwell_s_complete=("median_dwell_s_complete", "median"),
        median_frac_runs_complete=("frac_runs_complete", "median"),
        median_of_file_median_bound_occupancy=("median_track_bound_occupancy", "median"),
    )
    .reindex(DWELL_GENOS)
    .reset_index()
)

print(f"Whole-track {DWELL_STATE} dwell | {SEC_PER_WINDOW_STEP:.4f} s/window-step")
print(
    "N bound runs:",
    len(dwell_track_runs),
    "| complete:",
    int(dwell_track_runs["complete"].sum()),
    "| left-cens:",
    int(dwell_track_runs["left_censored"].sum()),
    "| right-cens:",
    int(dwell_track_runs["right_censored"].sum()),
)
display(summary_tr)

# --- KM-ish survival curves: P(dwell >= k) from all visits (naive) ---
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6), constrained_layout=True)
palette = {g: PLOT_GROUP_COLOR_MAP.get(g, "#888888") for g in DWELL_GENOS}

# 1) box: file median dwell (all)
sns.boxplot(
    data=per_file_tr,
    x="_switch_scope",
    y="median_dwell_windows_all",
    order=DWELL_GENOS,
    hue="_switch_scope",
    palette=palette,
    legend=False,
    ax=axes[0],
    showfliers=False,
)
sns.stripplot(
    data=per_file_tr,
    x="_switch_scope",
    y="median_dwell_windows_all",
    order=DWELL_GENOS,
    color="k",
    size=3,
    alpha=0.45,
    ax=axes[0],
)
axes[0].set_title("file median dwell (windows)\nall track bound visits")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)

# 2) box: complete-only
sub_c = per_file_tr.dropna(subset=["median_dwell_windows_complete"])
sns.boxplot(
    data=sub_c,
    x="_switch_scope",
    y="median_dwell_windows_complete",
    order=DWELL_GENOS,
    hue="_switch_scope",
    palette=palette,
    legend=False,
    ax=axes[1],
    showfliers=False,
)
sns.stripplot(
    data=sub_c,
    x="_switch_scope",
    y="median_dwell_windows_complete",
    order=DWELL_GENOS,
    color="k",
    size=3,
    alpha=0.45,
    ax=axes[1],
)
axes[1].set_title("file median dwell (windows)\ncomplete visits only")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)

# 3) survival S(k) = fraction of bound runs with dwell >= k
ax = axes[2]
k_max = int(np.nanpercentile(dwell_track_runs["dwell_windows"], 95))
k_max = max(k_max, 2)
ks = np.arange(1, k_max + 1)
for geno in DWELL_GENOS:
    durs = dwell_track_runs.loc[
        dwell_track_runs["_switch_scope"].astype(str) == geno, "dwell_windows"
    ].to_numpy(dtype=float)
    if len(durs) == 0:
        continue
    surv = np.array([(durs >= k).mean() for k in ks])
    ax.plot(ks, surv, label=geno, color=palette.get(geno, None), lw=2)
ax.set_xlabel("dwell ≥ k windows")
ax.set_ylabel("fraction of bound visits")
ax.set_title("bound dwell survival (pooled runs)")
ax.legend(fontsize=8, frameon=False)
ax.set_ylim(0, 1.02)

save_b = PLASTICITY_PLOTS_DIR / "FINAL_dwell_bound_WHOLETRACK_fileMEDIAN.svg"
fig.savefig(save_b, bbox_inches="tight")
plt.show()
print("saved:", save_b.name)

# Side-by-side citation line if A was run
if "summary_sw" in dir():
    print("\nCompare A (SW-local, all runs) vs B (track, all visits) — median-of-file-medians (windows):")
    for geno in DWELL_GENOS:
        a = summary_sw.loc[summary_sw["_switch_scope"] == geno, "median_of_file_median_dwell_windows_all"]
        b = summary_tr.loc[summary_tr["_switch_scope"] == geno, "median_of_file_median_dwell_windows_all"]
        av = float(a.iloc[0]) if len(a) else np.nan
        bv = float(b.iloc[0]) if len(b) else np.nan
        print(f"  {geno}: SW={av:.3f}  track={bv:.3f}")

print("DONE 8c-dwell-B (whole-track).")

## FINAL FIG: allelic series (direct / indirect) — 4 switches, boxplots + pairwise p

In [ ]:
# =============================================================================
# 8c2. Allelic series (DIRECT vs INDIRECT) — 4 switches, boxplots + pairwise p
# =============================================================================
# Separate plots for:
#   direct:   20H20S, 56H20S, 77H20S, 150H20S
#   indirect: 20H20S, 20H56S, 20H77S, 20H150S
# Transitions: bound→confined, bound→exploratory, confined→exploratory, exploratory→bound
# Unit = filename mean (SW mean within L → equal mean across kept lengths).
# Stats: Mann–Whitney U (two-sided) for every geno pair within that series;
#        Bonferroni within each panel.
from itertools import combinations
from scipy import stats as scipy_stats

sns.set_style("whitegrid")

UNIT_COL = "filename"
FOCUS_PAIRS = [
    ("bound", "confined"),
    ("bound", "exploratory"),
    ("confined", "exploratory"),
    ("exploratory", "bound"),
]
SERIES = {
    "direct": ["20H20S", "56H20S", "77H20S", "150H20S"],
    "indirect": ["20H20S", "20H56S", "20H77S", "20H150S"],
}
SERIES_COLORS = {
    "direct": {
        "20H20S": "#0072B2",
        "56H20S": "#56B4E9",
        "77H20S": "#009E73",
        "150H20S": "#D55E00",
    },
    "indirect": {
        "20H20S": "#0072B2",
        "20H56S": "#56B4E9",
        "20H77S": "#009E73",
        "20H150S": "#D55E00",
    },
}

# None = all lengths; or e.g. [20] / [10, 20]
LENGTHS_KEEP = None

# Per-panel y-limits keyed by (series, from, to). None → autoscale.
# Also falls back to YLIMS_SHARED[(from, to)] if series-specific missing.
YLIMS_SHARED = {
    ("bound", "confined"): None,
    ("bound", "exploratory"): None,
    ("confined", "exploratory"): None,
    ("exploratory", "bound"): None,
}
YLIMS = {}  # e.g. YLIMS[("direct", "bound", "exploratory")] = (0, 0.15)
# Example shared:
# YLIMS_SHARED[("bound", "exploratory")] = (0, 0.15)

ALPHA = 0.05
BONFERRONI = True

ALL_GENOS = sorted({g for gs in SERIES.values() for g in gs})
if UNIT_COL not in sw_overall.columns:
    raise KeyError(f"{UNIT_COL!r} missing from sw_overall")

sw_focus = sw_overall.loc[sw_overall["_switch_scope"].astype(str).isin(ALL_GENOS)].copy()
if LENGTHS_KEEP is not None:
    sw_focus = sw_focus.loc[
        sw_focus["superwindow_length"].astype(int).isin({int(L) for L in LENGTHS_KEEP})
    ].copy()
    print("LENGTHS_KEEP:", LENGTHS_KEEP, "| n SW:", len(sw_focus))
else:
    print("LENGTHS_KEEP: all | n SW:", len(sw_focus))

per_sw = plasticity._per_superwindow_transition_rows(
    sw_focus,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
)
meta = sw_focus[["superwindow_id", UNIT_COL]].drop_duplicates("superwindow_id")
per_sw = per_sw.merge(meta, on="superwindow_id", how="left")

keep_from = {a for a, _ in FOCUS_PAIRS}
keep_to = {b for _, b in FOCUS_PAIRS}
per_sw = per_sw.loc[
    per_sw["from_state"].astype(str).isin(keep_from)
    & per_sw["to_state"].astype(str).isin(keep_to)
].copy()

per_unit_L = (
    per_sw.groupby(
        ["_switch_scope", UNIT_COL, "superwindow_length", "from_state", "to_state"],
        dropna=False,
    )["transition_probability_per_superwindow"]
    .mean()
    .reset_index(name="p_unit_L")
)
per_unit = (
    per_unit_L.groupby(
        ["_switch_scope", UNIT_COL, "from_state", "to_state"],
        dropna=False,
    )["p_unit_L"]
    .mean()
    .reset_index(name="p_unit")
)
plot_df = per_unit.dropna(subset=["p_unit"]).copy()
plot_df["_switch_scope"] = plot_df["_switch_scope"].astype(str)

print("N files by geno:")
print(plot_df.groupby("_switch_scope")[UNIT_COL].nunique().reindex(ALL_GENOS))

stat_rows = []
Ltag = (
    "allL"
    if LENGTHS_KEEP is None
    else ("L" + ",".join(str(L) for L in LENGTHS_KEEP))
)


def _sig(p_adj: float) -> str:
    if not np.isfinite(p_adj):
        return "n/a"
    if p_adj < 0.001:
        return "***"
    if p_adj < 0.01:
        return "**"
    if p_adj < ALPHA:
        return "*"
    return "ns"


for series_name, genos in SERIES.items():
    colors = SERIES_COLORS[series_name]
    n_pair_tests = len(list(combinations(genos, 2)))
    bonf_factor = n_pair_tests if BONFERRONI else 1
    print(f"\n######## SERIES: {series_name.upper()} | genos={genos} ########")

    for a, b in FOCUS_PAIRS:
        pair_lab = f"{a} → {b}"
        pair_tag = f"{a}_to_{b}"
        sub = plot_df.loc[
            (plot_df["_switch_scope"].isin(genos))
            & (plot_df["from_state"].astype(str) == a)
            & (plot_df["to_state"].astype(str) == b)
        ].copy()
        if sub.empty:
            print(f"skip empty: {series_name} | {pair_lab}")
            continue
        sub["_switch_scope"] = pd.Categorical(
            sub["_switch_scope"], categories=genos, ordered=True
        )

        print(
            f"\n=== [{series_name}] {pair_lab} | Mann–Whitney U (two-sided)"
            f"{' + Bonferroni x' + str(bonf_factor) if BONFERRONI else ''} ==="
        )
        vals_by = {}
        for g in genos:
            v = sub.loc[sub["_switch_scope"] == g, "p_unit"].to_numpy(dtype=float)
            v = v[np.isfinite(v)]
            vals_by[g] = v
            if len(v):
                print(f"  {g}: n={len(v)} mean={np.mean(v):.4f} median={np.median(v):.4f}")
            else:
                print(f"  {g}: n=0")

        samples = [vals_by[g] for g in genos if len(vals_by[g]) >= 1]
        if len(samples) >= 2:
            try:
                h, p_kw = scipy_stats.kruskal(*samples)
                print(f"  Kruskal–Wallis: H={h:.3f} p={p_kw:.3g}")
            except ValueError as e:
                print(f"  Kruskal–Wallis skipped: {e}")
                p_kw = np.nan
        else:
            p_kw = np.nan

        for g1, g2 in combinations(genos, 2):
            v1, v2 = vals_by[g1], vals_by[g2]
            if len(v1) < 1 or len(v2) < 1:
                p_raw, p_adj, u = np.nan, np.nan, np.nan
            else:
                u, p_raw = scipy_stats.mannwhitneyu(v1, v2, alternative="two-sided")
                p_adj = min(float(p_raw) * bonf_factor, 1.0)
            sig = _sig(p_adj)
            u_s = f"{u:.1f}" if np.isfinite(u) else "n/a"
            p_s = f"{p_raw:.3g}" if np.isfinite(p_raw) else "n/a"
            pa_s = f"{p_adj:.3g}" if np.isfinite(p_adj) else "n/a"
            print(f"  {g1} vs {g2}: U={u_s}  p={p_s}  p_adj={pa_s}  {sig}")

            stat_rows.append(
                {
                    "series": series_name,
                    "transition": pair_lab,
                    "geno_a": g1,
                    "geno_b": g2,
                    "n_a": len(v1),
                    "n_b": len(v2),
                    "U": u,
                    "p": p_raw,
                    "p_adj": p_adj,
                    "sig": sig,
                    "kruskal_p": p_kw,
                }
            )

        ylim = YLIMS.get((series_name, a, b), YLIMS_SHARED.get((a, b)))
        if ylim is None:
            ymax = float(np.nanmax(sub["p_unit"].to_numpy(dtype=float)))
            ylim = (0.0, max(ymax * 1.12, 1e-3))

        fig, ax = plt.subplots(figsize=(5.4, 4.8))
        sns.boxplot(
            data=sub,
            x="_switch_scope",
            y="p_unit",
            order=genos,
            hue="_switch_scope",
            hue_order=genos,
            palette=colors,
            ax=ax,
            width=0.7,
            linewidth=0.9,
            fliersize=0,
            legend=False,
            boxprops=dict(alpha=0.9),
            medianprops=dict(color="k", linewidth=1.2),
        )
        sns.stripplot(
            data=sub,
            x="_switch_scope",
            y="p_unit",
            order=genos,
            color="0.15",
            ax=ax,
            size=3.8,
            alpha=0.5,
            jitter=0.18,
        )
        ax.set_xlabel("")
        ax.set_ylabel("P(next | current)", fontsize=11)
        ax.set_title(
            f"{pair_lab} | {series_name}\n{Ltag}",
            fontsize=12,
            fontweight="bold",
        )
        ax.set_ylim(*ylim)
        ax.tick_params(axis="x", labelsize=9, rotation=15)
        sns.despine(ax=ax)
        fig.tight_layout()
        out = (
            PLASTICITY_PLOTS_DIR
            / f"FINAL_box_allelic_{series_name}_{pair_tag}_{Ltag}_{TAG}_{SWITCH_QUANTITY}.svg"
        )
        fig.savefig(out, bbox_inches="tight")
        plt.show()
        print("saved:", out.name)

stats_df = pd.DataFrame(stat_rows)
display(stats_df)
print("\nYlims: YLIMS[(series, from, to)] or YLIMS_SHARED[(from, to)] = (ymin, ymax)")
print("DONE 8c2 (allelic series).")

## FINAL FIG: allelic series (direct / indirect) — mean bars + file dots + pairwise p

In [ ]:
# =============================================================================
# 8c3. Allelic series (DIRECT vs INDIRECT) — mean bars + file dots + pairwise p
# =============================================================================
# Same setup as 8c2, but bar = mean across files (± SEM) with per-file dots on top.
from itertools import combinations
from scipy import stats as scipy_stats

sns.set_style("whitegrid")

UNIT_COL = "filename"
FOCUS_PAIRS = [
    ("bound", "confined"),
    ("bound", "exploratory"),
    ("confined", "exploratory"),
    ("exploratory", "bound"),
]
SERIES = {
    "direct": ["20H20S", "56H20S", "77H20S", "150H20S"],
    "indirect": ["20H20S", "20H56S", "20H77S", "20H150S"],
}
SERIES_COLORS = {
    "direct": {
        "20H20S": "#0072B2",
        "56H20S": "#56B4E9",
        "77H20S": "#009E73",
        "150H20S": "#D55E00",
    },
    "indirect": {
        "20H20S": "#0072B2",
        "20H56S": "#56B4E9",
        "20H77S": "#009E73",
        "20H150S": "#D55E00",
    },
}

# None = all lengths; or e.g. [20] / [10, 20]
LENGTHS_KEEP = None

YLIMS_SHARED = {
    ("bound", "confined"): None,
    ("bound", "exploratory"): None,
    ("confined", "exploratory"): None,
    ("exploratory", "bound"): None,
}
YLIMS = {}  # e.g. YLIMS[("direct", "bound", "exploratory")] = (0, 0.15)

ALPHA = 0.05
BONFERRONI = True

ALL_GENOS = sorted({g for gs in SERIES.values() for g in gs})
if UNIT_COL not in sw_overall.columns:
    raise KeyError(f"{UNIT_COL!r} missing from sw_overall")

sw_focus = sw_overall.loc[sw_overall["_switch_scope"].astype(str).isin(ALL_GENOS)].copy()
if LENGTHS_KEEP is not None:
    sw_focus = sw_focus.loc[
        sw_focus["superwindow_length"].astype(int).isin({int(L) for L in LENGTHS_KEEP})
    ].copy()
    print("LENGTHS_KEEP:", LENGTHS_KEEP, "| n SW:", len(sw_focus))
else:
    print("LENGTHS_KEEP: all | n SW:", len(sw_focus))

per_sw = plasticity._per_superwindow_transition_rows(
    sw_focus,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
)
meta = sw_focus[["superwindow_id", UNIT_COL]].drop_duplicates("superwindow_id")
per_sw = per_sw.merge(meta, on="superwindow_id", how="left")

keep_from = {a for a, _ in FOCUS_PAIRS}
keep_to = {b for _, b in FOCUS_PAIRS}
per_sw = per_sw.loc[
    per_sw["from_state"].astype(str).isin(keep_from)
    & per_sw["to_state"].astype(str).isin(keep_to)
].copy()

per_unit_L = (
    per_sw.groupby(
        ["_switch_scope", UNIT_COL, "superwindow_length", "from_state", "to_state"],
        dropna=False,
    )["transition_probability_per_superwindow"]
    .mean()
    .reset_index(name="p_unit_L")
)
per_unit = (
    per_unit_L.groupby(
        ["_switch_scope", UNIT_COL, "from_state", "to_state"],
        dropna=False,
    )["p_unit_L"]
    .mean()
    .reset_index(name="p_unit")
)
plot_df = per_unit.dropna(subset=["p_unit"]).copy()
plot_df["_switch_scope"] = plot_df["_switch_scope"].astype(str)

print("N files by geno:")
print(plot_df.groupby("_switch_scope")[UNIT_COL].nunique().reindex(ALL_GENOS))

stat_rows = []
Ltag = (
    "allL"
    if LENGTHS_KEEP is None
    else ("L" + ",".join(str(L) for L in LENGTHS_KEEP))
)


def _sig(p_adj: float) -> str:
    if not np.isfinite(p_adj):
        return "n/a"
    if p_adj < 0.001:
        return "***"
    if p_adj < 0.01:
        return "**"
    if p_adj < ALPHA:
        return "*"
    return "ns"


for series_name, genos in SERIES.items():
    colors = SERIES_COLORS[series_name]
    n_pair_tests = len(list(combinations(genos, 2)))
    bonf_factor = n_pair_tests if BONFERRONI else 1
    print(f"\n######## SERIES: {series_name.upper()} | genos={genos} ########")
    x = np.arange(len(genos), dtype=float)

    for a, b in FOCUS_PAIRS:
        pair_lab = f"{a} → {b}"
        pair_tag = f"{a}_to_{b}"
        sub = plot_df.loc[
            (plot_df["_switch_scope"].isin(genos))
            & (plot_df["from_state"].astype(str) == a)
            & (plot_df["to_state"].astype(str) == b)
        ].copy()
        if sub.empty:
            print(f"skip empty: {series_name} | {pair_lab}")
            continue

        print(
            f"\n=== [{series_name}] {pair_lab} | Mann–Whitney U (two-sided)"
            f"{' + Bonferroni x' + str(bonf_factor) if BONFERRONI else ''} ==="
        )
        vals_by = {}
        means, sems = [], []
        for g in genos:
            v = sub.loc[sub["_switch_scope"] == g, "p_unit"].to_numpy(dtype=float)
            v = v[np.isfinite(v)]
            vals_by[g] = v
            if len(v):
                means.append(float(np.mean(v)))
                sems.append(float(np.std(v, ddof=1) / np.sqrt(len(v))) if len(v) > 1 else 0.0)
                print(f"  {g}: n={len(v)} mean={means[-1]:.4f} median={np.median(v):.4f}")
            else:
                means.append(np.nan)
                sems.append(0.0)
                print(f"  {g}: n=0")

        samples = [vals_by[g] for g in genos if len(vals_by[g]) >= 1]
        if len(samples) >= 2:
            try:
                h, p_kw = scipy_stats.kruskal(*samples)
                print(f"  Kruskal–Wallis: H={h:.3f} p={p_kw:.3g}")
            except ValueError as e:
                print(f"  Kruskal–Wallis skipped: {e}")
                p_kw = np.nan
        else:
            p_kw = np.nan

        for g1, g2 in combinations(genos, 2):
            v1, v2 = vals_by[g1], vals_by[g2]
            if len(v1) < 1 or len(v2) < 1:
                p_raw, p_adj, u = np.nan, np.nan, np.nan
            else:
                u, p_raw = scipy_stats.mannwhitneyu(v1, v2, alternative="two-sided")
                p_adj = min(float(p_raw) * bonf_factor, 1.0)
            sig = _sig(p_adj)
            u_s = f"{u:.1f}" if np.isfinite(u) else "n/a"
            p_s = f"{p_raw:.3g}" if np.isfinite(p_raw) else "n/a"
            pa_s = f"{p_adj:.3g}" if np.isfinite(p_adj) else "n/a"
            print(f"  {g1} vs {g2}: U={u_s}  p={p_s}  p_adj={pa_s}  {sig}")
            stat_rows.append(
                {
                    "series": series_name,
                    "transition": pair_lab,
                    "geno_a": g1,
                    "geno_b": g2,
                    "n_a": len(v1),
                    "n_b": len(v2),
                    "U": u,
                    "p": p_raw,
                    "p_adj": p_adj,
                    "sig": sig,
                    "kruskal_p": p_kw,
                }
            )

        ylim = YLIMS.get((series_name, a, b), YLIMS_SHARED.get((a, b)))
        if ylim is None:
            ymax = float(np.nanmax(sub["p_unit"].to_numpy(dtype=float)))
            ylim = (0.0, max(ymax * 1.12, 1e-3))

        fig, ax = plt.subplots(figsize=(5.4, 4.8))
        ax.bar(
            x,
            means,
            width=0.72,
            color=[colors[g] for g in genos],
            edgecolor="white",
            linewidth=0.7,
            alpha=0.9,
            zorder=2,
        )
        ax.errorbar(
            x, means, yerr=sems, fmt="none", ecolor="0.2", elinewidth=1.0, capsize=3.5, zorder=4
        )
        rng = np.random.default_rng(
            UNIT_RANDOM_STATE + (hash((series_name, pair_tag)) % 10_000)
        )
        for xi, g in enumerate(genos):
            vals = vals_by[g]
            if vals.size == 0:
                continue
            jj = rng.uniform(-0.14, 0.14, size=vals.size)
            ax.scatter(
                np.full(vals.size, xi) + jj,
                vals,
                s=22,
                color="0.1",
                alpha=0.55,
                edgecolors="none",
                zorder=3,
            )

        ax.set_xticks(x)
        ax.set_xticklabels(genos, fontsize=9, rotation=15)
        ax.set_ylabel("P(next | current)", fontsize=11)
        ax.set_xlabel("")
        ax.set_title(
            f"{pair_lab} | {series_name}\nmean ± SEM + files | {Ltag}",
            fontsize=12,
            fontweight="bold",
        )
        ax.set_ylim(*ylim)
        ax.yaxis.grid(True, alpha=0.25)
        ax.set_axisbelow(True)
        sns.despine(ax=ax)
        fig.tight_layout()
        out = (
            PLASTICITY_PLOTS_DIR
            / f"FINAL_bar_allelic_{series_name}_{pair_tag}_{Ltag}_{TAG}_{SWITCH_QUANTITY}.svg"
        )
        fig.savefig(out, bbox_inches="tight")
        plt.show()
        print("saved:", out.name)

stats_df_bars = pd.DataFrame(stat_rows)
display(stats_df_bars)
print("\nYlims: YLIMS[(series, from, to)] or YLIMS_SHARED[(from, to)] = (ymin, ymax)")
print("DONE 8c3 (allelic bars + dots).")

In [ ]:
# =============================================================================
# 8c2. Four switches — boxplots only, per-panel ylims, pairwise p-tests
# =============================================================================
# Transitions: bound→confined, bound→exploratory, confined→exploratory, exploratory→bound
# Unit = filename mean (SW mean within L → equal mean across kept lengths).
# Stats: Mann–Whitney U (two-sided) for every geno pair; Bonferroni within each panel.
from itertools import combinations
from scipy import stats as scipy_stats

sns.set_style("whitegrid")

UNIT_COL = "filename"
FOCUS_GENOS = ["20H20S", "150H20S", "20H150S"]
FOCUS_PAIRS = [
    ("bound", "confined"),
    ("bound", "exploratory"),
    ("confined", "exploratory"),
    ("exploratory", "bound"),
]
GENO_COLORS = {
    "20H20S": "#0072B2",
    "150H20S": "#009E73",
    "20H150S": "#D55E00",
}

# None = all lengths in sw_overall; or e.g. [20] / [19, 20, 21]
LENGTHS_KEEP = None

# Per-panel y-limits: None → autoscale; or (ymin, ymax)
YLIMS = {
    ("bound", "confined"): None,
    ("bound", "exploratory"): None,
    ("confined", "exploratory"): None,
    ("exploratory", "bound"): None,
}
# Example:
# YLIMS[("bound", "exploratory")] = (0, 0.15)

ALPHA = 0.05
BONFERRONI = True  # correct within each panel (3 pairwise tests)

if UNIT_COL not in sw_overall.columns:
    raise KeyError(f"{UNIT_COL!r} missing from sw_overall")

sw_focus = sw_overall.loc[sw_overall["_switch_scope"].astype(str).isin(FOCUS_GENOS)].copy()
if LENGTHS_KEEP is not None:
    sw_focus = sw_focus.loc[
        sw_focus["superwindow_length"].astype(int).isin({int(L) for L in LENGTHS_KEEP})
    ].copy()
    print("LENGTHS_KEEP:", LENGTHS_KEEP, "| n SW:", len(sw_focus))
else:
    print("LENGTHS_KEEP: all | n SW:", len(sw_focus))

per_sw = plasticity._per_superwindow_transition_rows(
    sw_focus,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
)
meta = sw_focus[["superwindow_id", UNIT_COL]].drop_duplicates("superwindow_id")
per_sw = per_sw.merge(meta, on="superwindow_id", how="left")

keep_from = {a for a, _ in FOCUS_PAIRS}
keep_to = {b for _, b in FOCUS_PAIRS}
per_sw = per_sw.loc[
    per_sw["from_state"].astype(str).isin(keep_from)
    & per_sw["to_state"].astype(str).isin(keep_to)
].copy()

per_unit_L = (
    per_sw.groupby(
        ["_switch_scope", UNIT_COL, "superwindow_length", "from_state", "to_state"],
        dropna=False,
    )["transition_probability_per_superwindow"]
    .mean()
    .reset_index(name="p_unit_L")
)
per_unit = (
    per_unit_L.groupby(
        ["_switch_scope", UNIT_COL, "from_state", "to_state"],
        dropna=False,
    )["p_unit_L"]
    .mean()
    .reset_index(name="p_unit")
)
per_unit["_switch_scope"] = pd.Categorical(
    per_unit["_switch_scope"].astype(str), categories=FOCUS_GENOS, ordered=True
)
plot_df = per_unit.dropna(subset=["p_unit"]).copy()

print("N files by geno:")
print(plot_df.groupby("_switch_scope", observed=True)[UNIT_COL].nunique())

stat_rows = []
n_pair_tests = len(list(combinations(FOCUS_GENOS, 2)))
bonf_factor = n_pair_tests if BONFERRONI else 1

for a, b in FOCUS_PAIRS:
    pair_lab = f"{a} → {b}"
    pair_tag = f"{a}_to_{b}"
    sub = plot_df.loc[
        (plot_df["from_state"].astype(str) == a) & (plot_df["to_state"].astype(str) == b)
    ].copy()
    if sub.empty:
        print(f"skip empty: {pair_lab}")
        continue

    # ----- pairwise Mann–Whitney -----
    print(f"\n=== {pair_lab} | Mann–Whitney U (two-sided)"
          f"{' + Bonferroni x'+str(bonf_factor) if BONFERRONI else ''} ===")
    vals_by = {
        g: sub.loc[sub["_switch_scope"] == g, "p_unit"].to_numpy(dtype=float)
        for g in FOCUS_GENOS
    }
    for g in FOCUS_GENOS:
        v = vals_by[g]
        v = v[np.isfinite(v)]
        vals_by[g] = v
        print(f"  {g}: n={len(v)} mean={np.mean(v):.4f} median={np.median(v):.4f}")

    # omnibus
    samples = [vals_by[g] for g in FOCUS_GENOS if len(vals_by[g]) >= 1]
    if len(samples) >= 2 and all(len(s) >= 1 for s in samples):
        try:
            h, p_kw = scipy_stats.kruskal(*samples)
            print(f"  Kruskal–Wallis: H={h:.3f} p={p_kw:.3g}")
        except ValueError as e:
            print(f"  Kruskal–Wallis skipped: {e}")
            p_kw = np.nan
    else:
        p_kw = np.nan

    for g1, g2 in combinations(FOCUS_GENOS, 2):
        v1, v2 = vals_by[g1], vals_by[g2]
        if len(v1) < 1 or len(v2) < 1:
            p_raw, p_adj, u = np.nan, np.nan, np.nan
        else:
            u, p_raw = scipy_stats.mannwhitneyu(v1, v2, alternative="two-sided")
            p_adj = min(float(p_raw) * bonf_factor, 1.0)
        sig = (
            "***" if p_adj < 0.001 else
            "**" if p_adj < 0.01 else
            "*" if p_adj < ALPHA else
            "ns"
        )
        print(
            f"  {g1} vs {g2}: U={u:.1f}  p={p_raw:.3g}  "
            f"p_adj={p_adj:.3g}  {sig}"
        )
        stat_rows.append(
            {
                "transition": pair_lab,
                "geno_a": g1,
                "geno_b": g2,
                "n_a": len(v1),
                "n_b": len(v2),
                "U": u,
                "p": p_raw,
                "p_adj": p_adj,
                "sig": sig,
                "kruskal_p": p_kw,
            }
        )

    # ----- boxplot -----
    ylim = YLIMS.get((a, b))
    if ylim is None:
        ymax = float(np.nanmax(sub["p_unit"].to_numpy(dtype=float)))
        ylim = (0.0, max(ymax * 1.12, 1e-3))

    fig, ax = plt.subplots(figsize=(4.6, 4.8))
    sns.boxplot(
        data=sub,
        x="_switch_scope",
        y="p_unit",
        order=FOCUS_GENOS,
        hue="_switch_scope",
        hue_order=FOCUS_GENOS,
        palette=GENO_COLORS,
        ax=ax,
        width=0.65,
        linewidth=0.9,
        fliersize=0,
        legend=False,
        boxprops=dict(alpha=0.9),
        medianprops=dict(color="k", linewidth=1.2),
    )
    sns.stripplot(
        data=sub,
        x="_switch_scope",
        y="p_unit",
        order=FOCUS_GENOS,
        color="0.15",
        ax=ax,
        size=4.0,
        alpha=0.5,
        jitter=0.18,
    )
    ax.set_xlabel("")
    ax.set_ylabel("P(next | current)", fontsize=11)
    Ltag = (
        "allL"
        if LENGTHS_KEEP is None
        else ("L" + ",".join(str(L) for L in LENGTHS_KEEP))
    )
    ax.set_title(f"{pair_lab}\n{Ltag}", fontsize=12, fontweight="bold")
    ax.set_ylim(*ylim)
    ax.tick_params(axis="x", labelsize=9)
    sns.despine(ax=ax)
    fig.tight_layout()
    out = PLASTICITY_PLOTS_DIR / f"FINAL_box4_{pair_tag}_{Ltag}_{TAG}_{SWITCH_QUANTITY}.svg"
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    print("saved:", out.name)

stats_df = pd.DataFrame(stat_rows)
display(stats_df)
print("\nEdit YLIMS[(from, to)] = (ymin, ymax) and re-run to lock axes.")
print("DONE 8c2.")

## FINAL FIG: per-switch bars with **one dot per superwindow length**

In [ ]:
# =============================================================================
# 8d. Per switch: bars = equal-L mean; dots = one point per superwindow length
# =============================================================================
# Same geno trio / no-transport pairs as 8c.
# For each length L: geno-level mean of per-superwindow P (set1 within-L).
# Bar height = equal mean of those per-L values. Dots = the per-L values.
sns.set_style("whitegrid")

UNIT_COL = "filename"  # unused for dots here; length is the unit
FOCUS_GENOS = ["20H20S", "150H20S", "20H150S"]
FOCUS_STATES = ["bound", "confined", "exploratory"]
FOCUS_PAIRS = [(a, b) for a in FOCUS_STATES for b in FOCUS_STATES if a != b]
GENO_COLORS = {"20H20S": "#0072B2", "150H20S": "#009E73", "20H150S": "#D55E00"}

sw_focus = sw_overall.loc[sw_overall["_switch_scope"].astype(str).isin(FOCUS_GENOS)].copy()
per_sw = plasticity._per_superwindow_transition_rows(
    sw_focus,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
)
per_sw = per_sw.loc[
    per_sw["from_state"].astype(str).isin(FOCUS_STATES)
    & per_sw["to_state"].astype(str).isin(FOCUS_STATES)
    & (per_sw["from_state"].astype(str) != per_sw["to_state"].astype(str))
].copy()

# (geno, L, pair) → mean over superwindows
per_L = (
    per_sw.groupby(
        ["_switch_scope", "superwindow_length", "from_state", "to_state"],
        dropna=False,
    )["transition_probability_per_superwindow"]
    .mean()
    .reset_index(name="p_L")
)
per_L["_switch_scope"] = per_L["_switch_scope"].astype(str)
per_L["superwindow_length"] = per_L["superwindow_length"].astype(int)

print("Lengths present:", sorted(per_L["superwindow_length"].unique()))
print("N (geno×L) rows:", len(per_L))

x = np.arange(len(FOCUS_GENOS), dtype=float)
rng = np.random.default_rng(UNIT_RANDOM_STATE)

for a, b in FOCUS_PAIRS:
    pair_lab = f"{a} → {b}"
    pair_tag = f"{a}_to_{b}"
    sub = per_L.loc[
        (per_L["from_state"].astype(str) == a) & (per_L["to_state"].astype(str) == b)
    ].copy()
    if sub.empty:
        print("skip", pair_lab)
        continue

    means = []
    for geno in FOCUS_GENOS:
        vals = sub.loc[sub["_switch_scope"] == geno, "p_L"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        means.append(float(np.mean(vals)) if vals.size else np.nan)

    ymax = float(np.nanmax(sub["p_L"].to_numpy(dtype=float)))
    ymax = max(ymax * 1.12, 1e-3)

    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    ax.bar(
        x,
        means,
        width=0.72,
        color=[GENO_COLORS[g] for g in FOCUS_GENOS],
        edgecolor="white",
        linewidth=0.7,
        alpha=0.9,
        zorder=2,
        label="equal-L mean",
    )
    for xi, geno in enumerate(FOCUS_GENOS):
        g = sub.loc[sub["_switch_scope"] == geno].copy()
        vals = g["p_L"].to_numpy(dtype=float)
        lens = g["superwindow_length"].to_numpy(dtype=int)
        m = np.isfinite(vals)
        vals, lens = vals[m], lens[m]
        if vals.size == 0:
            continue
        jj = rng.uniform(-0.14, 0.14, size=vals.size)
        ax.scatter(
            np.full(vals.size, xi) + jj,
            vals,
            s=28,
            c=lens,
            cmap="viridis",
            alpha=0.85,
            edgecolors="0.2",
            linewidths=0.3,
            zorder=3,
        )
    # colourbar for length
    sm = plt.cm.ScalarMappable(
        cmap="viridis",
        norm=plt.Normalize(
            vmin=float(per_L["superwindow_length"].min()),
            vmax=float(per_L["superwindow_length"].max()),
        ),
    )
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("superwindow length", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(FOCUS_GENOS, fontsize=9)
    ax.set_ylabel("P(next | current)", fontsize=11)
    ax.set_title(f"{pair_lab}\nbars=equal-L mean · dots=each L", fontsize=12, fontweight="bold")
    ax.set_ylim(0, ymax)
    ax.yaxis.grid(True, alpha=0.25)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)
    fig.tight_layout()
    out = PLASTICITY_PLOTS_DIR / f"FINAL_bar_dotsPerL_{pair_tag}_20_150_{TAG}_{SWITCH_QUANTITY}.svg"
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    print("saved:", out.name)

print("DONE 8d.")

## Diagnostic: which fixed L / small L-range matches priors (150 vs 20)?

In [ ]:
# =============================================================================
# 8f. DIAGNOSTIC — paste-back dump: which L / L-range matches priors?
# =============================================================================
# Priors vs 20H20S (stabilised %):
#   DECREASE: confined→bound, exploratory→bound, exploratory→confined
#   INCREASE: bound→exploratory, confined→exploratory
# Genos scored: 150H20S and 20H150S (both must agree for a "full" hit).
# Estimand: mean_per_superwindow within each L; ranges = equal mean of those L's.
# Requires sw_overall from cell 6. Paste the printed block back.

CONTROL = "20H20S"
TEST_GENOS = ["150H20S", "20H150S"]
EPS = float(FAN_STABILIZE_EPS) if "FAN_STABILIZE_EPS" in globals() else 0.1

PRIOR_PAIRS = [
    (("confined", "bound"), "decrease"),
    (("exploratory", "bound"), "decrease"),
    (("exploratory", "confined"), "decrease"),
    (("bound", "exploratory"), "increase"),
    (("confined", "exploratory"), "increase"),
]

FOCUS_GENOS = [CONTROL] + TEST_GENOS
sw = sw_overall.loc[sw_overall["_switch_scope"].astype(str).isin(FOCUS_GENOS)].copy()

per_sw = plasticity._per_superwindow_transition_rows(
    sw,
    group_col="_switch_scope",
    state_order=SUPERWINDOW_STATE_ORDER,
    track_col="original_track_id",
    length_col="superwindow_length",
)
per_L = (
    per_sw.groupby(
        ["_switch_scope", "superwindow_length", "from_state", "to_state"],
        dropna=False,
    )["transition_probability_per_superwindow"]
    .mean()
    .reset_index(name="p")
)
per_L["_switch_scope"] = per_L["_switch_scope"].astype(str)
per_L["superwindow_length"] = per_L["superwindow_length"].astype(int)
per_L["from_state"] = per_L["from_state"].astype(str)
per_L["to_state"] = per_L["to_state"].astype(str)

lengths = sorted(per_L["superwindow_length"].unique())


def _p_lookup(df_L: pd.DataFrame, geno: str, a: str, b: str) -> float:
    sub = df_L.loc[
        (df_L["_switch_scope"] == geno)
        & (df_L["from_state"] == a)
        & (df_L["to_state"] == b),
        "p",
    ]
    if sub.empty:
        return np.nan
    return float(sub.iloc[0])


def _rel(p: float, p0: float) -> float:
    if not np.isfinite(p) or not np.isfinite(p0):
        return np.nan
    return 100.0 * (p - p0) / (p0 + EPS)


def _score_table(p_by_geno: dict) -> tuple[pd.DataFrame, int, int]:
    """p_by_geno[geno][(a,b)] = absolute P. Returns detail rows + n_ok + n_total."""
    rows = []
    n_ok = 0
    n_tot = 0
    p0 = p_by_geno.get(CONTROL, {})
    for (a, b), direction in PRIOR_PAIRS:
        p_ctrl = p0.get((a, b), np.nan)
        for geno in TEST_GENOS:
            p = p_by_geno.get(geno, {}).get((a, b), np.nan)
            r = _rel(p, p_ctrl)
            if direction == "decrease":
                ok = bool(np.isfinite(r) and r < 0)
            else:
                ok = bool(np.isfinite(r) and r > 0)
            n_tot += 1
            n_ok += int(ok)
            rows.append(
                {
                    "pair": f"{a}->{b}",
                    "prior": direction,
                    "geno": geno,
                    "abs": p,
                    "ctrl_abs": p_ctrl,
                    "rel_pct": r,
                    "ok": ok,
                }
            )
    return pd.DataFrame(rows), n_ok, n_tot


def _amalgamate(Ls: list[int]) -> dict:
    """Equal mean across lengths of per-L geno means."""
    sub = per_L.loc[per_L["superwindow_length"].isin(Ls)]
    if sub.empty:
        return {}
    avg = (
        sub.groupby(["_switch_scope", "from_state", "to_state"], dropna=False)["p"]
        .mean()
        .reset_index()
    )
    out = {g: {} for g in FOCUS_GENOS}
    for _, r in avg.iterrows():
        out[str(r["_switch_scope"])][(str(r["from_state"]), str(r["to_state"]))] = float(r["p"])
    return out


# ---- single lengths ----
single_scores = []
single_details = {}
for L in lengths:
    pmap = _amalgamate([L])
    det, n_ok, n_tot = _score_table(pmap)
    single_details[L] = det
    # also require BOTH genos ok on each pair for "both_agree"
    both = 0
    both_tot = len(PRIOR_PAIRS)
    for (a, b), direction in PRIOR_PAIRS:
        oks = det.loc[det["pair"] == f"{a}->{b}", "ok"].tolist()
        both += int(len(oks) == 2 and all(oks))
    single_scores.append(
        {
            "label": f"L={L}",
            "kind": "single",
            "lengths": [L],
            "n_ok": n_ok,
            "n_tot": n_tot,
            "frac": n_ok / n_tot if n_tot else np.nan,
            "n_pairs_both_genos_ok": both,
            "n_pairs": both_tot,
        }
    )

# ---- small contiguous ranges (width 3 and 5) ----
range_scores = []
range_details = {}
for width in (3, 5):
    for i in range(0, max(0, len(lengths) - width + 1)):
        Ls = lengths[i : i + width]
        key = f"L{Ls[0]}-{Ls[-1]}"
        pmap = _amalgamate(Ls)
        det, n_ok, n_tot = _score_table(pmap)
        range_details[key] = det
        both = 0
        for (a, b), direction in PRIOR_PAIRS:
            oks = det.loc[det["pair"] == f"{a}->{b}", "ok"].tolist()
            both += int(len(oks) == 2 and all(oks))
        range_scores.append(
            {
                "label": key,
                "kind": f"range_w{width}",
                "lengths": Ls,
                "n_ok": n_ok,
                "n_tot": n_tot,
                "frac": n_ok / n_tot if n_tot else np.nan,
                "n_pairs_both_genos_ok": both,
                "n_pairs": len(PRIOR_PAIRS),
            }
        )

score_df = pd.DataFrame(single_scores + range_scores).sort_values(
    ["n_pairs_both_genos_ok", "n_ok", "frac"], ascending=False
)

# ---- PASTE BLOCK ----
lines = []
lines.append("=" * 72)
lines.append("PRIOR LENGTH DIAGNOSTIC (paste back)")
lines.append("=" * 72)
lines.append(f"CONTROL={CONTROL} | TEST={TEST_GENOS} | EPS={EPS}")
lines.append(f"MODE={SUPERWINDOW_MODE if 'SUPERWINDOW_MODE' in globals() else '?'} | unit=mean_per_superwindow per L")
lines.append(f"LENGTHS_PRESENT={lengths}")
lines.append(
    "PRIORS: decrease=[confined->bound, exploratory->bound, exploratory->confined] "
    "increase=[bound->exploratory, confined->exploratory]"
)
lines.append("SCORE: n_ok = geno×pair hits (max 10); both = pairs where BOTH test genos hit (max 5)")
lines.append("-" * 72)
lines.append("TOP SETTINGS (by both-genos-ok, then n_ok):")
for _, r in score_df.head(12).iterrows():
    lines.append(
        f"  {r['label']:<12} kind={r['kind']:<10} "
        f"both={int(r['n_pairs_both_genos_ok'])}/{int(r['n_pairs'])} "
        f"hits={int(r['n_ok'])}/{int(r['n_tot'])} frac={r['frac']:.2f}"
    )

# detail dump for best 5 + classic L=15 if present
show_labels = list(score_df["label"].head(5))
if "L=15" in score_df["label"].to_numpy() and "L=15" not in show_labels:
    show_labels.append("L=15")

lines.append("-" * 72)
lines.append("DETAIL (rel% vs control; OK=prior direction):")
for lab in show_labels:
    if lab.startswith("L=") and lab[2:].isdigit():
        det = single_details[int(lab[2:])]
    else:
        det = range_details.get(lab)
        if det is None:
            continue
    lines.append(f"\n[{lab}]")
    for _, row in det.iterrows():
        flag = "OK" if row["ok"] else "FAIL"
        rp = row["rel_pct"]
        rp_s = "n/a" if not np.isfinite(rp) else f"{rp:+.1f}%"
        abs_s = "n/a" if not np.isfinite(row["abs"]) else f"{row['abs']:.4f}"
        c_s = "n/a" if not np.isfinite(row["ctrl_abs"]) else f"{row['ctrl_abs']:.4f}"
        lines.append(
            f"  {flag:4} {row['geno']:<8} {row['pair']:<28} "
            f"prior={row['prior']:<9} abs={abs_s} ctrl={c_s} rel={rp_s}"
        )

# per-length one-liner matrix for quick scan
lines.append("\n" + "-" * 72)
lines.append("PER-LENGTH ONE-LINERS (rel% ; want - - - + + for the five pairs, each geno):")
pair_order = [f"{a}->{b}" for (a, b), _ in PRIOR_PAIRS]
for L in lengths:
    det = single_details[L]
    bits = [f"L={L}"]
    for geno in TEST_GENOS:
        gbits = []
        for pair in pair_order:
            r = det.loc[(det["geno"] == geno) & (det["pair"] == pair), "rel_pct"]
            v = float(r.iloc[0]) if len(r) else np.nan
            gbits.append("n/a" if not np.isfinite(v) else f"{v:+.0f}")
        bits.append(f"{geno}[{','.join(gbits)}]")
    # both count
    sc = score_df.loc[score_df["label"] == f"L={L}"].iloc[0]
    bits.append(f"both={int(sc['n_pairs_both_genos_ok'])}/5 hits={int(sc['n_ok'])}/10")
    lines.append("  " + " | ".join(bits))

lines.append("=" * 72)
lines.append("END DIAGNOSTIC")
lines.append("=" * 72)

text = "\n".join(lines)
print(text)

# also keep tables in notebook
display(score_df.head(20))

## FINAL FIG: curated galleries L=10 then L=20 (pooled, no transport; optional split by geno)

In [ ]:
# =============================================================================
# 9g. Curated galleries: L=10 then L=20 (NOT random; no transport)
# =============================================================================
# Default: pool ALL conditions into one row (black boxes).
# Set SPLIT_BY_CONDITION=True to restore geno-colored rows.
# Column strategy:
#   pure (bound/confined/exploratory) → 2 switches → 3 switches → many

GALLERY_LENGTHS = [10, 20]
SPLIT_BY_CONDITION = False  # True → one row per geno (colored); False → pooled (black)
GALLERY_GROUP_COL = PLOT_GROUP_COL
GALLERY_GROUP_ORDER = list(PLOT_GROUP_ORDER)
# No transport in pure slots OR in any selected sequence
GALLERY_STATES = [s for s in SUPERWINDOW_STATE_ORDER if str(s) != "transport"]
EXCLUDE_IF_CONTAINS = {"transport"}
TRACK_GALLERY_XY_SPAN_UM = 6.0
BORDER_COLOR_POOLED = "black"
NCOLS_GRID = 4  # wrap into a grid (not one long row) when pooled

N_SLOTS_SWITCH_2 = 2
N_SLOTS_SWITCH_3 = 2
N_SLOTS_SWITCH_MANY = 2
PURE_FRAC_MIN = 0.75
MANY_SWITCH_MIN = 4

# Sequence-panel cosmetics
SEQUENCE_SHOW_SUPERWINDOW_ID = False  # set True to annotate unique_id
SEQUENCE_SHOW_SLOT_TITLES = False     # hide "slot 0", …
SEQUENCE_SHOW_ROW_LABELS = False      # hide "r0", geno labels, …
SEQUENCE_X_TICK_INTERVAL = 5          # integer ticks every N positions
SEQUENCE_MARKER_SIZE_BY_L = {10: 160, 20: 70}  # smaller blobs for L=20
SEQUENCE_MARKER_SIZE_DEFAULT = 120

sw_all = (
    filtered_superwindow_df_for_visualization.to_pandas()
    if isinstance(filtered_superwindow_df_for_visualization, pl.DataFrame)
    else filtered_superwindow_df_for_visualization.copy()
)
sw_all = sw_all.loc[sw_all[GALLERY_GROUP_COL].isin(GALLERY_GROUP_ORDER)].copy()


def _seq_list(seq):
    return [str(x) for x in list(seq)]


def _has_excluded(seq) -> bool:
    return any(s in EXCLUDE_IF_CONTAINS for s in _seq_list(seq))


def _seq_stats(seq) -> dict:
    seq = _seq_list(seq)
    n = len(seq)
    if n == 0:
        return {
            "n_switches": 0,
            "dominant_state": None,
            "dominant_frac": 0.0,
            "n_unique_states": 0,
        }
    n_sw = int(sum(a != b for a, b in zip(seq[:-1], seq[1:])))
    counts = {}
    for s in seq:
        counts[s] = counts.get(s, 0) + 1
    dom, c = max(counts.items(), key=lambda kv: kv[1])
    return {
        "n_switches": n_sw,
        "dominant_state": dom,
        "dominant_frac": float(c) / float(n),
        "n_unique_states": len(counts),
    }


def _annotate(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    stats = out["state_sequence"].map(_seq_stats)
    out["n_switches"] = stats.map(lambda d: d["n_switches"])
    out["dominant_state"] = stats.map(lambda d: d["dominant_state"])
    out["dominant_frac"] = stats.map(lambda d: d["dominant_frac"])
    out["n_unique_states"] = stats.map(lambda d: d["n_unique_states"])
    out["has_excluded_state"] = out["state_sequence"].map(_has_excluded)
    return out


def _pick(sub: pd.DataFrame, n: int, *, sort_cols, ascending) -> pd.DataFrame:
    if sub.empty or n <= 0:
        return sub.iloc[0:0].copy()
    return sub.sort_values(sort_cols, ascending=ascending).head(int(n)).copy()


def select_curated_from_pool(pool: pd.DataFrame, states) -> pd.DataFrame:
    """Ordered picks from a pooled (or single-geno) dataframe — no transport."""
    gdf = pool.loc[~pool["has_excluded_state"]].copy()
    if gdf.empty:
        return gdf
    used = set()
    picked_rows = []

    def _take(cand, n, sort_cols, ascending):
        cand = cand.loc[~cand["superwindow_id"].isin(used)]
        take = _pick(cand, n, sort_cols=sort_cols, ascending=ascending)
        for sid in take["superwindow_id"].tolist():
            used.add(sid)
        return take

    for st in states:
        cand = gdf.loc[
            (gdf["dominant_state"].astype(str) == str(st))
            & (gdf["dominant_frac"] >= PURE_FRAC_MIN)
        ]
        if cand.empty:
            cand = gdf.loc[gdf["dominant_state"].astype(str) == str(st)]
        take = _take(cand, 1, ["dominant_frac", "n_switches"], [False, True])
        if not take.empty:
            take = take.copy()
            take["select_reason"] = f"pure:{st}"
            picked_rows.append(take)

    for n_sw, n_slots, label in (
        (2, N_SLOTS_SWITCH_2, "switches:2"),
        (3, N_SLOTS_SWITCH_3, "switches:3"),
    ):
        take = _take(
            gdf.loc[gdf["n_switches"] == n_sw],
            n_slots,
            ["n_unique_states", "dominant_frac"],
            [False, True],
        )
        if not take.empty:
            take = take.copy()
            take["select_reason"] = label
            picked_rows.append(take)

    take = _take(
        gdf.loc[gdf["n_switches"] >= MANY_SWITCH_MIN],
        N_SLOTS_SWITCH_MANY,
        ["n_switches", "n_unique_states"],
        [False, False],
    )
    if not take.empty:
        take = take.copy()
        take["select_reason"] = "switches:many"
        picked_rows.append(take)

    if not picked_rows:
        return gdf.iloc[0:0].copy()
    out = pd.concat(picked_rows, ignore_index=True)
    out["gallery_slot"] = np.arange(len(out))
    return out


def select_curated_gallery(df_L: pd.DataFrame) -> tuple[pd.DataFrame, list, str, dict]:
    """
    Returns sampled_df, group_order, group_col_for_plot, group_colors.
    """
    if SPLIT_BY_CONDITION:
        parts = []
        for g in GALLERY_GROUP_ORDER:
            gdf = df_L.loc[df_L[GALLERY_GROUP_COL].astype(str) == str(g)]
            picked = select_curated_from_pool(gdf, GALLERY_STATES)
            if picked.empty:
                continue
            parts.append(picked)
        if not parts:
            raise ValueError("No curated superwindows selected.")
        sampled = pd.concat(parts, ignore_index=True)
        sampled[GALLERY_GROUP_COL] = pd.Categorical(
            sampled[GALLERY_GROUP_COL], categories=list(GALLERY_GROUP_ORDER), ordered=True
        )
        sampled = sampled.sort_values([GALLERY_GROUP_COL, "gallery_slot"]).reset_index(drop=True)
        # re-number slots within each geno (already set in select_curated_from_pool)
        return (
            sampled,
            list(GALLERY_GROUP_ORDER),
            GALLERY_GROUP_COL,
            dict(PLOT_GROUP_COLOR_MAP),
        )

    # Pooled across conditions
    sampled = select_curated_from_pool(df_L, GALLERY_STATES)
    if sampled.empty:
        raise ValueError("No curated superwindows selected (pooled).")
    sampled = sampled.copy()
    sampled["_gallery_row"] = "curated"
    # Title uses this; keep reason visible, borders forced black
    sampled["_gallery_label"] = sampled["select_reason"].astype(str)
    group_colors = {lab: BORDER_COLOR_POOLED for lab in sampled["_gallery_label"].unique()}
    group_colors["curated"] = BORDER_COLOR_POOLED
    return sampled, ["curated"], "_gallery_row", group_colors


def plot_curated_for_length(L: int):
    df_L = sw_all.loc[sw_all["superwindow_length"].astype(int) == int(L)].copy()
    if df_L.empty:
        print(f"No superwindows at L={L}")
        return None
    df_L = _annotate(df_L)
    n_before = len(df_L)
    n_excl = int(df_L["has_excluded_state"].sum())
    print(f"L={L}: {n_before} SW, excluded {n_excl} with {sorted(EXCLUDE_IF_CONTAINS)}")

    sampled, group_order, group_col_plot, group_colors = select_curated_gallery(df_L)
    mode = "by_condition" if SPLIT_BY_CONDITION else "pooled_black"
    plot_df = sampled.copy()

    if SPLIT_BY_CONDITION:
        ncols = int(plot_df.groupby(group_col_plot)["gallery_slot"].max().max()) + 1
        seq_df = plot_df
        seq_group_col = group_col_plot
        seq_group_order = group_order
        label_col = GALLERY_GROUP_COL
        label_colors = group_colors
    else:
        ncols = int(NCOLS_GRID)
        plot_df = plot_df.sort_values("gallery_slot").reset_index(drop=True)
        plot_df["_grid_row"] = [f"r{i // ncols}" for i in range(len(plot_df))]
        plot_df["gallery_slot"] = [i % ncols for i in range(len(plot_df))]
        seq_df = plot_df
        seq_group_col = "_grid_row"
        seq_group_order = list(dict.fromkeys(plot_df["_grid_row"].tolist()))
        label_col = "_gallery_label"
        label_colors = {lab: BORDER_COLOR_POOLED for lab in plot_df[label_col].astype(str).unique()}

    print(f"\n===== L={L} | mode={mode} | grid {ncols} cols | n={len(plot_df)} =====")
    show_cols = [
        c
        for c in [
            GALLERY_GROUP_COL,
            "_grid_row",
            "gallery_slot",
            "select_reason",
            "dominant_state",
            "dominant_frac",
            "n_switches",
            "superwindow_id",
        ]
        if c in plot_df.columns
    ]
    display(plot_df[show_cols])

    tag = f"curated_L{L}_{mode}_{TAG}"

    marker_size = float(SEQUENCE_MARKER_SIZE_BY_L.get(int(L), SEQUENCE_MARKER_SIZE_DEFAULT))
    fig_seq, _, _ = plasticity.plot_ordered_superwindow_sequences(
        seq_df,
        group_col=seq_group_col,
        state_order=[s for s in SUPERWINDOW_STATE_ORDER if str(s) not in EXCLUDE_IF_CONTAINS],
        state_colors=SUPERWINDOW_STATE_COLOR_MAP,
        group_order=seq_group_order,
        n_per_group=ncols,
        line_color="k",
        show_superwindow_id=SEQUENCE_SHOW_SUPERWINDOW_ID,
        show_slot_titles=SEQUENCE_SHOW_SLOT_TITLES,
        show_row_labels=SEQUENCE_SHOW_ROW_LABELS,
        marker_size=marker_size,
        x_tick_interval=SEQUENCE_X_TICK_INTERVAL,
        save_path=PLASTICITY_PLOTS_DIR / f"FINAL_sequence_gallery_{tag}.svg",
        title=f"Curated sequences | L={L} | {mode} | {ncols}-col grid | no transport",
    )
    plt.show()

    sw_ids = plot_df["superwindow_id"].tolist()
    df_map = df_L.merge(
        plot_df[["superwindow_id", label_col]],
        on="superwindow_id",
        how="inner",
        suffixes=("", "_lab"),
    )

    window_state_map_df = superwindows.build_window_state_map_from_superwindows(
        df_map,
        sw_ids,
        cluster_col=label_col,
        group_col=label_col,
        visual_state_col="superwindow_window_state_for_visualization",
    )
    gallery_instant = superwindows.map_window_states_to_instant(
        filtered_instant_for_visualization,
        window_state_map_df,
        visual_state_col="superwindow_window_state_for_visualization",
        group_col=label_col,
        pixel_size_um=PIXELSIZE_MICRONS,
    )
    fig, _ = superwindows.plot_state_colored_track_gallery(
        gallery_instant,
        sw_ids,
        state_col="superwindow_window_state_for_visualization",
        state_colors=SUPERWINDOW_STATE_COLOR_MAP,
        cluster_col=label_col,
        cluster_colors=label_colors,
        cluster_order=None,
        ncols=ncols,
        xy_span_um=TRACK_GALLERY_XY_SPAN_UM,
        show_superwindow_id=SEQUENCE_SHOW_SUPERWINDOW_ID,
        show_cluster_in_title=True,  # keep select_reason / geno label; IDs follow toggle above
        save_path=PLASTICITY_PLOTS_DIR / f"FINAL_track_gallery_{tag}.svg",
    )
    plt.show()
    print("saved:", tag)
    return {"sampled": plot_df, "fig_seq": fig_seq, "fig": fig, "ncols": ncols}


print("SPLIT_BY_CONDITION =", SPLIT_BY_CONDITION)
results_by_L = {}
for L in GALLERY_LENGTHS:
    results_by_L[L] = plot_curated_for_length(L)

print("\nStrategy columns:")
print("  [pure bound | pure confined | pure exploratory |")
print(f"   2-sw ×{N_SLOTS_SWITCH_2} | 3-sw ×{N_SLOTS_SWITCH_3} | many (≥{MANY_SWITCH_MIN}) ×{N_SLOTS_SWITCH_MANY}]")
print("Set SPLIT_BY_CONDITION=True to restore geno rows / colored borders.")
print("DONE curated galleries.")

In [ ]:
# =============================================================================
# 9h. Paired sequence | track grid (L=10 and L=20 separately)
# =============================================================================
# Run 9g first. Sequence (narrow) | track (wide). A4-width figure; tight gaps.
from matplotlib.gridspec import GridSpec

GALLERY_LENGTHS = [10, 20]
EXAMPLES_PER_ROW = 4

# A4 portrait usable area
A4_WIDTH_MM, A4_HEIGHT_MM = 210.0, 297.0
PAGE_MARGIN_MM = 12.0
A4_USABLE_WIDTH_IN = (A4_WIDTH_MM - 2 * PAGE_MARGIN_MM) / 25.4
A4_USABLE_HEIGHT_IN = (A4_HEIGHT_MM - 2 * PAGE_MARGIN_MM) / 25.4
FIG_WIDTH_IN = A4_USABLE_WIDTH_IN

# Tight page chrome
LEFT, RIGHT, TOP, BOTTOM = 0.05, 0.995, 0.96, 0.02
WSPACE = 0.035          # between examples
HSPACE = 0.06           # between rows
PAIR_WSPACE = 0.03      # between seq and track
# Sequence narrow, track takes most of the example cell
SEQ_TRACK_WIDTH_RATIO = (1,1)#(0.55, 1.45)

MIN_FONT_PT = 6
SEQUENCE_X_TICK_INTERVAL = 5
# Smaller blobs
SEQUENCE_MARKER_SIZE_BY_L = {10: 36, 20: 16}
SEQUENCE_MARKER_SIZE_DEFAULT = 24
TRACK_XY_SPAN_UM = 6.0
SCALE_BAR_UM = 1.0
BORDER_COLOR = "black"
EXCLUDE_IF_CONTAINS = {"transport"}
SEQ_STATES = [s for s in SUPERWINDOW_STATE_ORDER if str(s) not in EXCLUDE_IF_CONTAINS]


def _layout(n_examples: int, examples_per_row: int):
    """Track-led sizing: example width from A4; height from track square side."""
    n_rows = int(np.ceil(n_examples / examples_per_row))
    usable_w = FIG_WIDTH_IN * (RIGHT - LEFT)
    # account for wspace between example columns (~(n-1)/n fraction)
    example_w = usable_w / examples_per_row
    # track fraction of example
    r_seq, r_tr = SEQ_TRACK_WIDTH_RATIO
    track_w = example_w * (r_tr / (r_seq + r_tr))
    # track panel ~ square → row height ≈ track width
    row_h = track_w * 1.02
    fig_h = (n_rows * row_h) / max(TOP - BOTTOM, 0.5)
    if fig_h > A4_USABLE_HEIGHT_IN:
        # clamp to one A4 page height; keep width, accept slightly shorter rows
        fig_h = A4_USABLE_HEIGHT_IN
        row_h = fig_h * (TOP - BOTTOM) / n_rows
    return n_rows, example_w, track_w, fig_h


def _ensure_pool_for_L(L: int) -> pd.DataFrame:
    if "results_by_L" in globals() and results_by_L.get(L) is not None:
        df = results_by_L[L]["sampled"].copy()
        if "select_reason" in df.columns:
            reason_order = []
            states = GALLERY_STATES if "GALLERY_STATES" in globals() else SEQ_STATES
            for st in states:
                reason_order.append(f"pure:{st}")
            reason_order += ["switches:2", "switches:3", "switches:many"]
            df["_reason_rank"] = df["select_reason"].map(
                {r: i for i, r in enumerate(reason_order)}
            ).fillna(99)
            df = df.sort_values(["_reason_rank"]).reset_index(drop=True)
        return df
    df_L = sw_all.loc[sw_all["superwindow_length"].astype(int) == int(L)].copy()
    df_L = _annotate(df_L)
    sampled = select_curated_from_pool(df_L, SEQ_STATES)
    if sampled.empty:
        raise ValueError(f"No curated SWs at L={L}. Run 9g first.")
    return sampled.reset_index(drop=True)


def _draw_sequence(ax, sequence, marker_size: float):
    seq = [str(x) for x in list(sequence)]
    states = list(SEQ_STATES)
    for t, state in enumerate(seq):
        if state not in states:
            continue
        y = states.index(state)
        ax.scatter(
            t, y, s=marker_size,
            c=[SUPERWINDOW_STATE_COLOR_MAP.get(state, "lightgray")],
            edgecolors="k", linewidths=0.45, zorder=5,
        )
        if t < len(seq) - 1:
            nxt = str(seq[t + 1])
            if nxt in states:
                ax.plot([t, t + 1], [y, states.index(nxt)], color="k", lw=0.7, alpha=0.45, zorder=3)
    ax.set_yticks(range(len(states)))
    ax.set_yticklabels(states, fontsize=MIN_FONT_PT)
    ax.set_xlim(-0.4, max(len(seq) - 0.6, 0.5))
    ax.set_ylim(-0.45, len(states) - 0.55)
    step = int(SEQUENCE_X_TICK_INTERVAL)
    ticks = list(range(0, max(len(seq), 1), step))
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(t) for t in ticks], fontsize=MIN_FONT_PT)
    ax.tick_params(axis="both", labelsize=MIN_FONT_PT, length=1.5, pad=0.8)
    ax.grid(True, axis="x", alpha=0.18, lw=0.4)
    for spine in ax.spines.values():
        spine.set_color(BORDER_COLOR)
        spine.set_linewidth(1.0)


def _draw_track(ax, track: pd.DataFrame, span_um: float):
    sort_cols = [c for c in ["position_in_superwindow", "frame"] if c in track.columns]
    if sort_cols:
        track = track.sort_values(sort_cols)
    x = track["x_um"].to_numpy(dtype=float)
    y = track["y_um"].to_numpy(dtype=float)
    states = track["superwindow_window_state_for_visualization"].astype(str).to_numpy()
    for i in range(max(0, len(track) - 1)):
        ax.plot(
            x[i:i + 2], y[i:i + 2],
            color=SUPERWINDOW_STATE_COLOR_MAP.get(states[i], "lightgray"),
            lw=1.6, solid_capstyle="round",
        )
    ax.scatter(
        x, y, s=4,
        c=[SUPERWINDOW_STATE_COLOR_MAP.get(s, "lightgray") for s in states],
        edgecolors="none", zorder=4,
    )
    xm = float((np.nanmin(x) + np.nanmax(x)) / 2)
    ym = float((np.nanmin(y) + np.nanmax(y)) / 2)
    ax.set_xlim(xm - span_um / 2, xm + span_um / 2)
    ax.set_ylim(ym - span_um / 2, ym + span_um / 2)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(BORDER_COLOR)
        spine.set_linewidth(1.2)
    x0 = xm - span_um / 2 + 0.07 * span_um
    y0 = ym - span_um / 2 + 0.07 * span_um
    ax.plot([x0, x0 + SCALE_BAR_UM], [y0, y0], color="k", lw=1.5, solid_capstyle="butt")
    ax.text(
        x0 + SCALE_BAR_UM / 2, y0 + 0.035 * span_um,
        f"{SCALE_BAR_UM:g} µm", ha="center", va="bottom",
        fontsize=MIN_FONT_PT, color="k",
    )


def plot_paired_grid_for_length(L: int):
    sampled = _ensure_pool_for_L(L)
    sampled = sampled.loc[
        ~sampled["state_sequence"].map(lambda s: any(str(x) in EXCLUDE_IF_CONTAINS for x in list(s)))
    ].reset_index(drop=True)
    n = len(sampled)
    if n == 0:
        print(f"L={L}: nothing to plot")
        return None

    df_L = sw_all.loc[sw_all["superwindow_length"].astype(int) == int(L)].copy()
    sw_ids = sampled["superwindow_id"].tolist()
    tmp = sampled[["superwindow_id"]].copy()
    tmp["_lab"] = "x"
    df_map = df_L.merge(tmp, on="superwindow_id", how="inner")
    window_state_map_df = superwindows.build_window_state_map_from_superwindows(
        df_map, sw_ids,
        cluster_col="_lab", group_col="_lab",
        visual_state_col="superwindow_window_state_for_visualization",
    )
    gallery_instant = superwindows.map_window_states_to_instant(
        filtered_instant_for_visualization,
        window_state_map_df,
        visual_state_col="superwindow_window_state_for_visualization",
        group_col="_lab",
        pixel_size_um=PIXELSIZE_MICRONS,
    )
    gi = (
        gallery_instant.to_pandas()
        if isinstance(gallery_instant, pl.DataFrame)
        else gallery_instant.copy()
    )
    gi["superwindow_id_str"] = gi["superwindow_id"].astype(str)

    n_rows, example_w, track_w, fig_h = _layout(n, EXAMPLES_PER_ROW)
    fig = plt.figure(figsize=(FIG_WIDTH_IN, fig_h))
    outer = GridSpec(
        n_rows, EXAMPLES_PER_ROW,
        figure=fig, wspace=WSPACE, hspace=HSPACE,
        left=LEFT, right=RIGHT, top=TOP, bottom=BOTTOM,
    )
    marker_size = float(SEQUENCE_MARKER_SIZE_BY_L.get(int(L), SEQUENCE_MARKER_SIZE_DEFAULT))

    for i in range(n):
        r, c = divmod(i, EXAMPLES_PER_ROW)
        cell = outer[r, c].subgridspec(
            1, 2, wspace=PAIR_WSPACE, width_ratios=list(SEQ_TRACK_WIDTH_RATIO)
        )
        ax_seq = fig.add_subplot(cell[0, 0])
        ax_tr = fig.add_subplot(cell[0, 1])

        row = sampled.iloc[i]
        _draw_sequence(ax_seq, row["state_sequence"], marker_size)

        sid = str(row["superwindow_id"])
        track = gi.loc[gi["superwindow_id_str"] == sid]
        if track.empty:
            ax_tr.axis("off")
        else:
            _draw_track(ax_tr, track, TRACK_XY_SPAN_UM)

        if c != 0:
            ax_seq.set_yticklabels([])

    for j in range(n, n_rows * EXAMPLES_PER_ROW):
        r, c = divmod(j, EXAMPLES_PER_ROW)
        fig.add_subplot(outer[r, c]).axis("off")

    fig.suptitle(
        f"L={L}  ·  {EXAMPLES_PER_ROW}/row  ·  A4 width",
        fontsize=max(MIN_FONT_PT, 8), y=0.99,
    )
    out = PLASTICITY_PLOTS_DIR / f"FINAL_paired_seq_track_L{L}_{TAG}.svg"
    # pad_inches=0.02 keeps export tight (less white around the grid)
    fig.savefig(out, bbox_inches="tight", pad_inches=0.02, transparent=True)
    plt.show()
    print(
        f"L={L}: n={n} | rows={n_rows} | example_w≈{example_w:.2f}in | "
        f"track_w≈{track_w:.2f}in | fig {FIG_WIDTH_IN:.2f}×{fig_h:.2f}in | {out.name}"
    )
    return fig


print(
    f"A4 usable {A4_USABLE_WIDTH_IN:.2f}×{A4_USABLE_HEIGHT_IN:.2f} in | "
    f"EXAMPLES_PER_ROW={EXAMPLES_PER_ROW} | ratio seq:track={SEQ_TRACK_WIDTH_RATIO}"
)
paired_figs = {}
for L in GALLERY_LENGTHS:
    paired_figs[L] = plot_paired_grid_for_length(L)
print("DONE 9h.")

## Citation-ready numbers

In [ ]:
# =============================================================================
# 10. Results dump (locked settings)
# =============================================================================
print("MODE:", SUPERWINDOW_MODE)
print("SW_UNIT_POLICY:", SW_UNIT_POLICY)
print("LENGTHS:", SUPERWINDOW_LENGTHS)
print("LENGTH_AGG_POLICY:", LENGTH_AGG_POLICY)
print("SWITCH_QUANTITY:", SWITCH_QUANTITY, "| DISPLAY:", FAN_DISPLAY_MODE, "eps=", FAN_STABILIZE_EPS)

disp = display_transition_df
disp_abs = display_transition_df_abs
abs_od = disp_abs.loc[~disp_abs["is_self_transition"]].copy()
abs_od["pair"] = abs_od["from_state"].astype(str) + " to " + abs_od["to_state"].astype(str)
display(
    abs_od.pivot_table(index="pair", columns="_switch_scope", values="display_value", aggfunc="mean")
    .reindex(columns=PLOT_GROUP_ORDER)
    .round(4)
)

delta = disp.loc[(~disp["is_self_transition"]) & (disp["_switch_scope"] != CONTROL_SCOPE)].copy()
delta["pair"] = delta["from_state"].astype(str) + " to " + delta["to_state"].astype(str)
delta_cols = [c for c in PLOT_GROUP_ORDER if c != CONTROL_SCOPE]
delta_wide = (
    delta.pivot_table(index="pair", columns="_switch_scope", values="display_value", aggfunc="mean")
    .reindex(columns=delta_cols)
)
ctrl_abs = abs_od.loc[abs_od["_switch_scope"] == CONTROL_SCOPE, ["pair", "display_value"]].rename(
    columns={"display_value": f"{CONTROL_SCOPE}_absolute"}
)
report = delta_wide.reset_index().merge(ctrl_abs, on="pair", how="left")
display(report.round(2))

print("CITATION-READY:")
for _, row in report.iterrows():
    bits = [f"{row['pair']}: {CONTROL_SCOPE} abs={row[f'{CONTROL_SCOPE}_absolute']:.3f}"]
    for col in delta_cols:
        v = row[col]
        bits.append(f"{col} {'n/a' if pd.isna(v) else f'{v:+.1f}%'}")
    print(" | ".join(bits))

## State-change movie (superwindow or whole track)

PNG-stack viewer for one **superwindow** or one **whole track**. The trail is colored by state; a state change inserts extra frames so you can see a flash (`lut_whiten`, `head_pulse`, and/or `track_gold`). Filter first for tracks that are mostly one state, mixed, or switch-heavy. Toggle raw-image overlay and Fiji LUTs at the top of the next cell.

In [ ]:
# =============================================================================
# 11. State-change superwindow / whole-track movie (PNG stack)
# =============================================================================
from IPython.display import Image, display


# --- view + pick ---
VIEW_MODE = "superwindow"          # "superwindow" or "whole_track"
UNIT_ID = None                     # force a superwindow_id / unique_id; None = use filter
N_MOVIES = 1

# --- composition filter (ignored when UNIT_ID is set) ---
TARGET_STATE = None                # e.g. "exploratory"; None = any dominant state
MIN_TARGET_FRACTION = 0.0
MAX_TARGET_FRACTION = 1.0
REQUIRE_STATES = None              # e.g. ["bound", "exploratory"] for a mix
EXCLUDE_STATES = None              # e.g. ["transport"]
MIN_N_DISTINCT_STATES = 2          # 2+ = mixed; 1 = allow pure
MAX_N_DISTINCT_STATES = None
MIN_N_SWITCHES = 1
MAX_N_SWITCHES = None
PREFER = "most_switches"           # most_switches | most_mixed | most_of_state | longest | fewest_switches | random
FILTER_GROUP_COL = PLOT_GROUP_COL
FILTER_GROUP_VALUES = None         # e.g. ["20H20S"]; None = all plotted groups
FILTER_LENGTH = DISPLAY_SUPERWINDOW_LENGTH if VIEW_MODE == "superwindow" else None
FILTER_RANDOM_STATE = 0

# --- movie look ---
OVERLAY_ON_RAW = True
BACKGROUND = "white"               # "white" or "black" when OVERLAY_ON_RAW is False
FOV_MODE = "track_crop"            # "track_crop" | "particle" | "full"
VIEW_ZOOM = 1.0                    # >1 zoom in, <1 zoom out (e.g. 0.7 / 1.4)
CROP_PAD_PX = 18                   # extra pixels around the track; raise to zoom out
BOX_SIZE = 48                      # particle-centered crop (pixels); raise to zoom out
TRAIL = "cumulative"               # "cumulative" or "current_state"
FLASH_MODES = ("head_pulse", "track_gold")  # add "lut_whiten" to flash the whole LUT
FLASH_HOLD_FRAMES = 6
LUT_WHITEN_AMOUNT = 0.55
HEAD_BASE_RADIUS_PX = 2.2
HEAD_PULSE_SCALE = 2.4
TRACK_LINEWIDTH = 2.8             # trail thickness
TRACK_SMOOTH = 1.2                 # Gaussian sigma in frames; 0 = off
TRACK_CURVE = 4                    # cubic samples per segment; 1 = straight
INVERT_RAW = False

# --- Fiji LUT (same loader as the rest of visualization.py) ---
USE_FIJI_LUT = True
FIJI_LUT_DIR = str(paths["fiji_lut_dir"])
FIJI_LUT_PREFIX = "JDM"
FIJI_LUT_NAME = "JDM_Ink Wash Yellow"
RAW_CMAP = "gray"
RAW_MASTER_DIR = config.MASTER
RAW_MOVIE_PATH = None              # override a specific tif if path inference fails

# --- output ---
WRITE_GIF = True
FPS = 16
SHOW_PREVIEW = True
MOVIE_DIR = PLASTICITY_PLOTS_DIR / "state_change_movies"

sw_for_movie = filtered_superwindow_df_for_visualization
if VIEW_MODE == "superwindow" and FILTER_LENGTH is not None:
    sw_for_movie = plasticity.subset_superwindows_by_length(sw_for_movie, FILTER_LENGTH)

chosen_lut = None
if USE_FIJI_LUT:
    try:
        jdm_luts, chosen_lut = raw_images.load_fiji_lut_collection(
            lut_dir=FIJI_LUT_DIR,
            prefix=FIJI_LUT_PREFIX,
            default_lut_name=FIJI_LUT_NAME,
        )
        print("Available Fiji LUTs:", list(jdm_luts))
        if FIJI_LUT_NAME in jdm_luts:
            chosen_lut = jdm_luts[FIJI_LUT_NAME]
    except Exception as exc:
        print(f"Fiji LUT load failed ({exc}); using matplotlib cmap {RAW_CMAP!r}")
        USE_FIJI_LUT = False

result = movies.make_state_change_track_movie(
    filtered_instant_for_visualization,
    superwindow_df=sw_for_movie,
    view_mode=VIEW_MODE,
    unit_id=UNIT_ID,
    target_state=TARGET_STATE,
    min_target_fraction=MIN_TARGET_FRACTION,
    max_target_fraction=MAX_TARGET_FRACTION,
    require_states=REQUIRE_STATES,
    exclude_states=EXCLUDE_STATES,
    min_n_distinct_states=MIN_N_DISTINCT_STATES,
    max_n_distinct_states=MAX_N_DISTINCT_STATES,
    min_n_switches=MIN_N_SWITCHES,
    max_n_switches=MAX_N_SWITCHES,
    group_col=FILTER_GROUP_COL,
    group_values=FILTER_GROUP_VALUES,
    length=FILTER_LENGTH,
    prefer=PREFER,
    n_select=N_MOVIES,
    random_state=FILTER_RANDOM_STATE,
    state_colors=SUPERWINDOW_STATE_COLOR_MAP,
    state_order=SUPERWINDOW_STATE_ORDER,
    overlay_on_raw=OVERLAY_ON_RAW,
    raw_movie_path=RAW_MOVIE_PATH,
    master_dir=RAW_MASTER_DIR,
    use_fiji_lut=USE_FIJI_LUT,
    fiji_lut=chosen_lut,
    fiji_lut_name=FIJI_LUT_NAME,
    fiji_lut_dir=FIJI_LUT_DIR,
    fiji_lut_prefix=FIJI_LUT_PREFIX,
    raw_cmap=RAW_CMAP,
    invert_raw=INVERT_RAW,
    background=BACKGROUND,
    fov_mode=FOV_MODE,
    crop_pad_px=CROP_PAD_PX,
    view_zoom=VIEW_ZOOM,
    box_size=BOX_SIZE,
    flash_modes=FLASH_MODES,
    flash_hold_frames=FLASH_HOLD_FRAMES,
    lut_whiten_amount=LUT_WHITEN_AMOUNT,
    head_base_radius_px=HEAD_BASE_RADIUS_PX,
    head_pulse_scale=HEAD_PULSE_SCALE,
    track_linewidth=TRACK_LINEWIDTH,
    track_smooth=TRACK_SMOOTH,
    track_curve=TRACK_CURVE,
    trail=TRAIL,
    pixel_size_um=PIXELSIZE_MICRONS,
    frame_interval=FRAME_INTERVAL,
    write_gif=WRITE_GIF,
    fps=FPS,
    preview_frames=SHOW_PREVIEW,
    show_plot=False,
    output_dir=MOVIE_DIR,
)

print("selected units:")
display(result["selected_df"].head(12))
print("ranked pool (top 12):")
display(result["summary_df"].head(12))

for movie in result["movies"]:
    print("PNG dir:", movie["output_dir"])
    print("GIF:", movie["gif_path"])
    print("state changes:", movie["n_state_changes"], "| pngs:", movie["n_pngs"])
    if SHOW_PREVIEW:
        for path in movie.get("preview_paths") or []:
            display(Image(filename=path))

In [ ]:
# =============================================================================
# 12. Batch GIFs — cycle TARGET_STATE × PREFER (~20 movies)
# =============================================================================
from IPython.display import Image, display


VIEW_MODE = "superwindow"
FILTER_LENGTH = DISPLAY_SUPERWINDOW_LENGTH if VIEW_MODE == "superwindow" else None
FILTER_GROUP_COL = PLOT_GROUP_COL
FILTER_GROUP_VALUES = None
FILTER_RANDOM_STATE = 0
MIN_TARGET_FRACTION = 0.0
MAX_TARGET_FRACTION = 1.0
REQUIRE_STATES = None
EXCLUDE_STATES = None
MIN_N_DISTINCT_STATES = 2
MAX_N_DISTINCT_STATES = None
MIN_N_SWITCHES = 1
MAX_N_SWITCHES = None

VIEW_ZOOM = 1.2
TRACK_SMOOTH = 1
TRACK_CURVE = 2
BACKGROUND = "white"
OVERLAY_ON_RAW = False
FOV_MODE = "track_crop"
CROP_PAD_PX = 18
BOX_SIZE = 48
TRAIL = "cumulative"
FLASH_MODES = ("track_gold",)
FLASH_HOLD_FRAMES = 6
LUT_WHITEN_AMOUNT = 0.55
HEAD_BASE_RADIUS_PX = 1.1
HEAD_PULSE_SCALE = 2.4
TRACK_LINEWIDTH = 1.0
INVERT_RAW = False
USE_FIJI_LUT = True
FIJI_LUT_DIR = str(paths["fiji_lut_dir"])
FIJI_LUT_PREFIX = "JDM"
FIJI_LUT_NAME = "JDM_Ink Wash Yellow"
RAW_CMAP = "gray"
RAW_MASTER_DIR = config.MASTER
RAW_MOVIE_PATH = None
FPS = 16
SHOW_PREVIEW = False
BATCH_DIR = PLASTICITY_PLOTS_DIR / "state_change_movies" / "batch_gifs_20"

TARGET_STATES = ["exploratory", "bound", "confined", "transport"]
N_BY_PREFER = {"most_of_state": 3, "most_mixed": 2}

sw_for_movie = filtered_superwindow_df_for_visualization
if VIEW_MODE == "superwindow" and FILTER_LENGTH is not None:
    sw_for_movie = plasticity.subset_superwindows_by_length(sw_for_movie, FILTER_LENGTH)

chosen_lut = None
if USE_FIJI_LUT:
    try:
        jdm_luts, chosen_lut = raw_images.load_fiji_lut_collection(
            lut_dir=FIJI_LUT_DIR,
            prefix=FIJI_LUT_PREFIX,
            default_lut_name=FIJI_LUT_NAME,
        )
        if FIJI_LUT_NAME in jdm_luts:
            chosen_lut = jdm_luts[FIJI_LUT_NAME]
    except Exception as exc:
        print(f"Fiji LUT load failed ({exc}); using {RAW_CMAP!r}")
        USE_FIJI_LUT = False

used_ids = []
batch_rows = []
n_ok = 0
n_fail = 0

for target_state in TARGET_STATES:
    for prefer, n_pick in N_BY_PREFER.items():
        print(f"\n=== {target_state} | {prefer} | n={n_pick} ===")
        try:
            result = movies.make_state_change_track_movie(
                filtered_instant_for_visualization,
                superwindow_df=sw_for_movie,
                view_mode=VIEW_MODE,
                unit_id=None,
                target_state=target_state,
                min_target_fraction=MIN_TARGET_FRACTION,
                max_target_fraction=MAX_TARGET_FRACTION,
                require_states=[target_state] if prefer == "most_mixed" else REQUIRE_STATES,
                exclude_states=EXCLUDE_STATES,
                min_n_distinct_states=MIN_N_DISTINCT_STATES,
                max_n_distinct_states=MAX_N_DISTINCT_STATES,
                min_n_switches=MIN_N_SWITCHES,
                max_n_switches=MAX_N_SWITCHES,
                group_col=FILTER_GROUP_COL,
                group_values=FILTER_GROUP_VALUES,
                length=FILTER_LENGTH,
                prefer=prefer,
                n_select=n_pick,
                random_state=FILTER_RANDOM_STATE,
                exclude_ids=used_ids,
                state_colors=SUPERWINDOW_STATE_COLOR_MAP,
                state_order=SUPERWINDOW_STATE_ORDER,
                overlay_on_raw=OVERLAY_ON_RAW,
                raw_movie_path=RAW_MOVIE_PATH,
                master_dir=RAW_MASTER_DIR,
                use_fiji_lut=USE_FIJI_LUT,
                fiji_lut=chosen_lut,
                fiji_lut_name=FIJI_LUT_NAME,
                fiji_lut_dir=FIJI_LUT_DIR,
                fiji_lut_prefix=FIJI_LUT_PREFIX,
                raw_cmap=RAW_CMAP,
                invert_raw=INVERT_RAW,
                background=BACKGROUND,
                fov_mode=FOV_MODE,
                crop_pad_px=CROP_PAD_PX,
                view_zoom=VIEW_ZOOM,
                box_size=BOX_SIZE,
                flash_modes=FLASH_MODES,
                flash_hold_frames=FLASH_HOLD_FRAMES,
                lut_whiten_amount=LUT_WHITEN_AMOUNT,
                head_base_radius_px=HEAD_BASE_RADIUS_PX,
                head_pulse_scale=HEAD_PULSE_SCALE,
                track_linewidth=TRACK_LINEWIDTH,
                track_smooth=TRACK_SMOOTH,
                track_curve=TRACK_CURVE,
                trail=TRAIL,
                pixel_size_um=PIXELSIZE_MICRONS,
                frame_interval=FRAME_INTERVAL,
                write_png=False,
                write_gif=True,
                fps=FPS,
                preview_frames=False,
                show_plot=False,
                output_dir=BATCH_DIR / f"{target_state}_{prefer}",
            )
        except Exception as exc:
            n_fail += 1
            print(f"SKIP {target_state} / {prefer}: {exc}")
            continue

        used_ids.extend(result["selected_ids"])
        for movie in result["movies"]:
            n_ok += 1
            batch_rows.append(
                {
                    "target_state": target_state,
                    "prefer": prefer,
                    "unit_id": movie["unit_id"],
                    "n_state_changes": movie["n_state_changes"],
                    "gif_path": movie["gif_path"],
                }
            )
            print("GIF:", movie["gif_path"])

print(f"\nDone. {n_ok} GIFs, {n_fail} skipped combos.")
print("Output:", BATCH_DIR)
display(pd.DataFrame(batch_rows))